# Digital Cash Management Forecast: load test

This notebook sets up the load test in this Dataiku project, a project of its own: it never reads or writes the
forecast project. It times the nightly forecast on fake data of the same shape as the real daily flows, with the
forecast project's own code (`cashflow.py`, `models.py`, `forecast.py`: the same files, copied by this notebook).

One scenario, **Run the load test**, run by hand. Its **step 1** (Scenarios > Run the load test > Steps > 1) sets
`tests`, run one after the other in one go. `tests` is a grid: each
number (the fake data's size: `clients`, `months`, `postings_per_month`, `clients_per_group`; and `workers`) is one
value or a list, and every combination is a test (here 3 x 2 x 3 = 18 tests). A list of tests, each with every
number, works too. For example:

```json
{"tests": {"clients": [100000, 400000, 800000], "months": 24, "postings_per_month": 60,
           "clients_per_group": [2000, 5000], "workers": [2, 4, 8]}}
```

Change them there, then Run. It runs, in 2 zones:

- **Data prep** (not timed): the fake daily flows of each size listed in `load_test_flows`, one subfolder per size,
  Parquet files laid out exactly like `daily_flows`; a size already there is kept, nothing is deleted (remove unused sizes by hand).
  Tests with the same size share its data.
- **Load test**: the placeholder model (`daily_catboost`: one CatBoost model, P10, EV and P90 together, the last 365
  days as inputs) fitted on 2 clients only (its forecasts mean nothing, its prediction time is real), then the nightly
  forecast of every fake client, `workers` groups of clients at a time, as production runs it, then its final load,
  timed: set by `FINAL_LOAD`, `"csv"` (a CSV file in the folder `load_test_forecast`: no table, no new dataset) or
  `"sql"` (the table `FINAL_SCHEMA`.`FINAL_TABLE` on `FINAL_CONNECTION`, by default `EDW_AI.LOAD_TEST_FINAL` on
  `cm_test_env`, its rows replaced, as production). The times: the forecast, the final load, and the share
  of each step of the forecast (reading, preparing,
  model, formatting, writing). Each test runs in its own process: its memory is measured on its own, and running out
  of memory is logged as a failed test (the next test still runs).
- **The report** (even when a test failed): the dashboard **Load test** and `latest_report.html` in the folder
  `load_test_report`. The number of clients is not a setting, so each client count has its own table and chart,
  and only the settings (workers, clients per group) are compared there. Then where the forecast's time goes, and
  each setting's time per 100,000 clients at each client count. Every test: one row of the
  dataset `load_test_log` (its `batch`: the run it was part of), and, when `LOG_SQL_CONNECTION` is set, the same rows
  added to a database table (`LOG_SQL_SCHEMA`.`LOG_SQL_TABLE`, by default `EDW_AI.LOAD_TEST_LOG` on the connection
  `cm_test_env`), with SQL statements only: no Dataiku dataset. This notebook creates the table when it is not there
  (and adds the columns a newer version of the code logs); if it can't, it prints the `CREATE TABLE` to run by hand.

**The final load `"sql"`:** this notebook makes the table with SQL when it is not there (the forecast's columns; it
adds the columns a newer version of the code writes); if it can't, it stops and prints the `CREATE TABLE` for a DBA.
On Oracle (the bank) the recipe writes the table with Oracle's own driver, `oracledb`, in batches with bind variables
(no Dataiku dataset: none is created), in one transaction: a failed load leaves the previous rows. At the bank, first:
- the package `oracledb` in the code env `CODE_ENV` (a Dataiku admin adds it; this notebook checks the env lists it);
- the connection's **Details readable by** (Administration > Connections > the connection > Security) includes your
  group: the recipe reads the connection's user and password from Dataiku to log in with `oracledb`;
- the connection's user may create the table (or a DBA makes it), delete its rows and insert into it.

On another database (PostgreSQL in the sandbox), a dataset `load_test_final` linked to the table is made here, and
the recipe fills it with Dataiku's writer.

**How to use it:** in a new empty project, import this notebook (Notebooks > New notebook > Import), fill in
`CODE_ENV`, `FINAL_LOAD` (and `FINAL_*` for `"sql"`), `LOG_SQL_*` (or `LOG_SQL_CONNECTION = ""`), and the scenario's first tests if you like, and run all cells (Cell > Run
All). It sets up the project, then (`RUN_NOW`) runs what the scenario runs and shows the result. Later: change the
tests in step 1 of the scenario, then Run. Run this notebook again after the forecast code changes, so the load test
times the new code: it keeps the tests set on the scenario.

In [ ]:
PROJECT_NAME = "Digital Cash Management Forecast - Load test"

# ── 1. PYTHON AND OUTPUT ────────────────────────────────────────────────────────────────────────────────────────────
CODE_ENV = "YOUR_CODE_ENV"               # the forecast project's code env (polars, catboost), e.g. "cashflow_forecast"
FINAL_LOAD = "csv"              # where each test's forecast is loaded at the end (timed):
                                #   "csv": a CSV file in the folder load_test_forecast (no table, no new dataset)
                                #   "sql": the table FINAL_SCHEMA.FINAL_TABLE on FINAL_CONNECTION, as production does
# Only for FINAL_LOAD = "sql": the table, made here with SQL if it is not there (the connection's user must be allowed
# to create it, delete its rows and insert). Oracle: written with oracledb (it must be in CODE_ENV, and the connection's
# "Details readable by" must include your group), no Dataiku dataset. Another database: through a dataset
# load_test_final linked to the table.
FINAL_CONNECTION = "cm_test_env"     # the Dataiku connection
FINAL_SCHEMA = "EDW_AI"              # the schema the table is in ("": the connection's own)
FINAL_TABLE = "LOAD_TEST_FINAL"      # the table
OUTPUT_CONNECTION = "filesystem_managed"         # datasets: load_test_log
OUTPUT_FOLDER_CONNECTION = "filesystem_folders"  # folders: the fake daily flows, the forecast, the reports
# A copy of load_test_log in a database table, added to at the end of each run (made here if it is not there; the
# connection's user must be allowed to create the table and insert into it). LOG_SQL_CONNECTION = "": no copy.
LOG_SQL_CONNECTION = "cm_test_env"     # the Dataiku connection
LOG_SQL_SCHEMA = "EDW_AI"              # the schema the table is in ("": the connection's own)
LOG_SQL_TABLE = "LOAD_TEST_LOG"        # the table

# ── 2. THE TESTS THE SCENARIO STARTS WITH ──────────────────────────────────────────────────────────────────────────
# The scenario "Run the load test" sets its tests in its step 1 (Scenarios > Run the load test > Steps > 1), run one
# after the other in one go. These are only its first tests, when the scenario is made: after that, change them
# there. Running this notebook again keeps them. TESTS is a grid: each number, or a list of numbers, and every
# combination is a test (here 3 x 2 x 3 = 18 tests). A list of tests, each with every number, works too.
#   clients              how many fake clients
#   months               months of daily flows per client, up to yesterday (at least 16)
#   postings_per_month   average postings (transactions) per client and month
#   clients_per_group    clients per group (a file per group and month)
#   workers              groups of clients forecast at a time (production: inference_workers)
# Tests with the same first 4 numbers share the same fake data (made once, not timed).
TESTS = {"clients": [100_000, 400_000, 800_000], "months": 24, "postings_per_month": 60,
         "clients_per_group": [2_000, 5_000], "workers": [2, 4, 8]}

RUN_NOW = True                  # at the end of this notebook: run what the scenario runs, with its step 1

## 1. Code environment, code, flow

Checks the code env has CatBoost, writes the project library, the 3 recipes in 2 zones (Data prep: `make_fake_data`; Load test: `run_load_test`, `load_test_report`) and the scenario, its tests in step 1.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())


def check_code_env(name, packages):
    """Check the code env exists and lists the packages. It is not created here: an admin creates it once."""
    request = (f"Ask a Dataiku admin to create the Python code env {name} (Python 3.9 or later) with these "
               f"packages: {', '.join(packages)}. Or set CODE_ENV to an existing env that has them.")
    if name not in [e["envName"] for e in client.list_code_envs()]:
        raise RuntimeError(f"Code env {name} not found. {request}")
    try:
        listed = " ".join(client.get_code_env("PYTHON", name).get_settings().get_required_packages(as_list=True))
    except Exception:          # no right to read the env settings: it is checked when the first recipe runs
        print(f"Code env {name} found (could not read its packages)")
        return
    missing = [p for p in packages if p.split("=")[0].split(">")[0] not in listed]
    if missing:
        raise RuntimeError(f"Code env {name} lacks {', '.join(missing)}. Ask a Dataiku admin to add "
                           f"{'it' if len(missing) == 1 else 'them'} to it (Administration > Code envs > {name} > "
                           "Packages to install, then Update). Or set CODE_ENV to an existing env that has them.")
    print(f"Code env {name} has {', '.join(packages)}")

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

The forecast reads 2 tables of the bank's warehouse, read-only, and creates or writes nothing there:
    DIM_CUSTOMER          EDW.DIM_CUSTOMER: the clients. The project keeps the business clients (companies and sole
                          proprietors) and only the columns below: personal fields are never read. Through the
                          database connection, a read-only SELECT of those columns (table_query), whole every run.
    FCT_STATEMENT_ENTRY   EDW.FCT_STATEMENT_ENTRY: the daily postings. The staging recipe sums them per client, day
                          and category (money in, money out) in Dataiku and adds the new days to daily_flows. Through
                          the database connection, the only query is a read-only SELECT of the columns below with a
                          date filter on VAL_DT (table_query): the last HISTORY_MONTHS months the first time (or after
                          the scenario Full rebuild), then only the days after the last day already in daily_flows.
Always through the database connection to the warehouse (the setup's EDW_CONNECTION: Oracle at the bank, read-only;
a local PostgreSQL with fake data on the test machine): the same code, only the connection differs. The schemas and
tables are the setup's settings (CLIENTS_*, POSTINGS_*); if a column or a code differs at the bank, only this file
changes."""
import numpy as np
import pandas as pd

# the tables the project reads. standard name: (its dataset in the Flow, the table by default). The setup links each
# dataset to the table its settings name (CLIENTS_SCHEMA, CLIENTS_TABLE, POSTINGS_SCHEMA, POSTINGS_TABLE); staging reads
# the dataset's own table, so only the settings change if the bank's names differ
TABLES = {
    "clients": ("DIM_CUSTOMER", "EDW.DIM_CUSTOMER"),
    "transactions": ("FCT_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
}
STAGED = ["clients", "daily_flows", "staging_log"]       # what the staging recipe writes

# Categories: each posting gets one from its transaction type (TRNS_TYP); a type not listed goes to OTHER.
# MOCK LIST: these are the sample data's types. At the bank, replace it with the real TRNS_TYP codes (ask the
# warehouse team for the list of codes and what each means); nothing else changes.
CATEGORY_OF_TYPE = {
    "COLLECTION": "CUSTOMER_RECEIPTS", "CARD_SETTLEMENT": "CUSTOMER_RECEIPTS", "CHEQUE": "CUSTOMER_RECEIPTS",
    "SALARY": "SALARIES", "RENT": "RENT", "SUPPLIER": "SUPPLIERS", "FINANCING": "FINANCING", "GOV": "GOVERNMENT",
    "UTILITY": "UTILITIES", "CARD": "CARDS", "FEE": "FEES",
}
OTHER = "OTHER"

# the columns read. standard name: warehouse (EDW) name
COLUMNS = {
    "clients": {"client_no": "CSTMR_CIF", "customer_type": "CSTMR_TYP", "line_of_business": "LOB_DSC",
                "legal_entity": "LGL_ENTY_DSC", "sub_legal_entity": "PRTY_SUB_LGL_DSC", "id_type": "PRTY_ID_TYP_DSC",
                "segment": "RM_SGMNT_NM", "sector": "SCTR_DSC", "business_sector": "BUS_SCTR_EN_NM",
                "industry": "INDST_EN_NM", "sub_industry": "SUB_INDST_EN_NM", "city": "CITY", "status": "STS_FLG"},
    # one row per posting. AMT_LCY: SAR, negative for money out (T24). TRNS_TYP: the transaction type, which gives
    # the category. RVRSL_MRKR: set on a reversing posting, which carries the reversed posting's TRNS_RFR
    "transactions": {"entry_id": "STMT_ENTRY_ID", "client_no": "CSTMR_CIF", "account_no": "ACNT_NMBR",
                     "value_date": "VAL_DT", "direction": "CRDT_DBT_FLG", "amount": "AMT_LCY", "txn_type": "TRNS_TYP",
                     "txn_ref": "TRNS_RFR", "reversal": "RVRSL_MRKR"},
}
# Which rows of DIM_CUSTOMER are read: 2 SQL conditions, applied in the warehouse (the bank's own query, checked on
# 2026-11-08: about 420k rows, one per customer). Column names in double quotes, so they work as they are on Oracle.
# The business clients: relationship manager code starting with 12, or the sector Sole Proprietorships
CLIENTS_BUSINESS = """ "PRIME_RM" LIKE '12%' OR "SCTR_DSC" = 'Sole Proprietorships' """
# DIM_CUSTOMER keeps history (about 520 million rows): each customer's current row is the one still open
CLIENTS_CURRENT_ROW = """ "END_DT" = DATE '2199-12-31' """
# FCT_STATEMENT_ENTRY is partitioned by month on its load date AS_OF_DT (not on the value date VAL_DT). Staging
# filters on it too, so Oracle reads only the load partitions that can hold the days it needs, not the whole table:
# AS_OF_DT from POSTINGS_LOAD_MARGIN_DAYS days before the first value day read. A posting loaded after its value
# date is always read; only one loaded more than that many days BEFORE its value date would be missed.
# None: no load-date filter (every partition is read).
POSTINGS_LOAD_DATE = "AS_OF_DT"
POSTINGS_LOAD_MARGIN_DAYS = 31
ACTIVE_STATUS = ("ACTIVE", "A")               # STS_FLG of an active client; an empty STS_FLG counts as active too
                                              # (an inactive one shows INACTIVE)
SOLE_PROPRIETOR = ("SOLE", "ESTABLISHMENT")   # words of LGL_ENTY_DSC, PRTY_SUB_LGL_DSC or SCTR_DSC for a business owned by
                                              # 1 person (the sector Sole Proprietorships, for example)


def read_columns(table):
    """The warehouse columns the project reads from a table."""
    return list(COLUMNS[table].values())


def to_standard(df, table):
    """A warehouse table with the standard column names and the project's values: client numbers as text, direction
    IN / OUT; for clients, status ACTIVE / CLOSED and sole_proprietor true / false. A missing column stops here
    with its name."""
    missing = [c for c in COLUMNS[table].values() if c not in df]
    if missing:
        raise ValueError(f"{TABLES[table][0]} has no column {', '.join(missing)}: check the warehouse table")
    out = pd.DataFrame({standard: df[column] for standard, column in COLUMNS[table].items()})
    out["client_no"] = out["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "direction" in out:
        out["direction"] = out["direction"].astype(str).str.strip().str.upper().str[0].map({"C": "IN", "D": "OUT"})
    if table == "clients":
        status_raw = out["status"].astype(str).str.strip().str.upper().replace(["NAN", "NONE", "NULL", ""], "")
        active = status_raw.isin(ACTIVE_STATUS) | status_raw.eq("")      # no status at all: active too
        out["status"] = np.where(active, "ACTIVE", "CLOSED")
        legal = (out["legal_entity"].fillna("").astype(str) + " " + out["sub_legal_entity"].fillna("").astype(str)
                 + " " + out["sector"].fillna("").astype(str))
        out["sole_proprietor"] = legal.str.upper().str.contains("|".join(SOLE_PROPRIETOR))
    return out


def one_row_per_client(clients):
    """One row per client number (the last one received). The warehouse already sends one per client
    (CLIENTS_CURRENT_ROW); staging warns if it did not."""
    return clients.drop_duplicates("client_no", keep="last").reset_index(drop=True)


def value_day(dates):
    """The day of each value date, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.normalize()


def table_query(table, name, since=None, end=None, count=False, limit=None):
    """The read-only query staging runs through the database connection: only the columns read (COLUMNS) of a table,
    named in the query, so it never depends on the column list Dataiku keeps for the dataset. name: the table's full
    quoted name. For the postings, only the value dates from since (None: the whole history) to before end; the value
    date comes back as text YYYY-MM-DD (TO_CHAR: Oracle and PostgreSQL), so no time zone can move it. count: the
    same rows counted (COUNT(*)). For the clients, only the business clients' current rows (clients_filter): the
    warehouse leaves the other customers and the old versions out, so far fewer rows travel."""
    date = COLUMNS["transactions"]["value_date"] if table == "transactions" else None
    columns = [f"TO_CHAR(\"{date}\", 'YYYY-MM-DD') AS \"{date}\"" if c == date else f'"{c}"'
               for c in COLUMNS[table].values()]
    query = f"SELECT {'COUNT(*) AS N' if count else ', '.join(columns)} FROM {name}"
    if table == "clients":
        query += f" WHERE {clients_filter()}"       # business clients' current rows only: the rest stays in Oracle
        if limit:                                   # only the first `limit` business clients (CLIENTS_LIMIT)
            query += f' ORDER BY "{COLUMNS["clients"]["client_no"]}" FETCH FIRST {int(limit)} ROWS ONLY'
    if date:
        where = [f"\"{date}\" < DATE '{end:%Y-%m-%d}'"]
        if since is not None:
            where.insert(0, f"\"{date}\" >= DATE '{since:%Y-%m-%d}'")
        query += f" WHERE {' AND '.join(where)}"
    return query


def clients_filter():
    """The SQL condition on DIM_CUSTOMER: the business clients (CLIENTS_BUSINESS), only each one's current row
    (CLIENTS_CURRENT_ROW). Used for the clients and, as a subquery, for the postings."""
    current = CLIENTS_CURRENT_ROW.strip()
    return f"({CLIENTS_BUSINESS.strip()})" + (f" AND ({current})" if current else "")


def _text(value):
    return "'" + str(value).replace("'", "''") + "'"


def daily_totals_query(postings, clients, since, end, limit=None):
    """The read-only query staging runs for the postings: the warehouse itself sums them into daily totals, so only
    one row per client, day and category comes back (columns client_no, day, category, money_in, money_out,
    postings), not every posting. postings, clients: the 2 tables' full quoted names. The rules (Oracle and
    PostgreSQL; NULL-safe, as Oracle treats '' as NULL):
    - 3 steps: the selected customers (DIM_CUSTOMER with clients_filter, one row each), the postings of the
      selected days (value dates from since, None: no lower bound, to before end), then the postings joined with
      the customers; with since, also only the load dates from
      POSTINGS_LOAD_MARGIN_DAYS before it (POSTINGS_LOAD_DATE: the partitions Oracle reads);
    - without reversed postings: a posting with a reversal marker (RVRSL_MRKR) carries the reversed posting's
      reference (TRNS_RFR), and every posting with that reference is left out;
    - without transfers between the client's own accounts: on the same client and reference, a credit and a debit
      that net to zero;
    - category from the transaction type (CATEGORY_OF_TYPE, else OTHER); money_in, money_out: the credits and the
      debits as positive SAR amounts (AMT_LCY); postings: how many.
    Window functions (one pass, no NOT IN / NOT EXISTS lookups) keep it fast on a big table."""
    c = COLUMNS["transactions"]
    dates = [f"\"{c['value_date']}\" < DATE '{end:%Y-%m-%d}'"]
    if since is not None:
        dates.insert(0, f"\"{c['value_date']}\" >= DATE '{since:%Y-%m-%d}'")
        if POSTINGS_LOAD_DATE and POSTINGS_LOAD_MARGIN_DAYS is not None:     # only the load partitions needed
            loaded = since - pd.Timedelta(days=POSTINGS_LOAD_MARGIN_DAYS)
            dates.append(f"\"{POSTINGS_LOAD_DATE}\" >= DATE '{loaded:%Y-%m-%d}'")
    cif = COLUMNS["clients"]["client_no"]
    # 1. the selected customers: one row per customer (DISTINCT), so the join below can never count a posting twice
    customers = f"SELECT DISTINCT \"{cif}\" AS c_cif FROM {clients} WHERE {clients_filter()}"
    if limit:                                       # the same first `limit` business clients as table_query
        customers += f" ORDER BY c_cif FETCH FIRST {int(limit)} ROWS ONLY"
    categories = " ".join(f"WHEN {_text(t)} THEN {_text(k)}" for t, k in CATEGORY_OF_TYPE.items())
    category = f"CASE UPPER(TRIM(p_typ)) {categories} ELSE {_text(OTHER)} END"
    return f"""WITH customers AS (
  {customers}
), postings AS (
  SELECT "{c['client_no']}" AS p_cif, TO_CHAR("{c['value_date']}", 'YYYY-MM-DD') AS p_day,
         UPPER(SUBSTR(TRIM("{c['direction']}"), 1, 1)) AS p_dir, "{c['amount']}" AS p_amt,
         "{c['txn_type']}" AS p_typ, "{c['txn_ref']}" AS p_ref, "{c['reversal']}" AS p_rev, "{c['entry_id']}" AS p_id
  FROM {postings}
  WHERE {' AND '.join(dates)}
), p AS (
  SELECT postings.* FROM postings JOIN customers ON customers.c_cif = postings.p_cif
), marked AS (
  SELECT p.*, MAX(CASE WHEN p_ref IS NOT NULL AND LENGTH(TRIM(p_rev)) > 0 AND TRIM(p_rev) NOT IN ('nan', 'None')
                       THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref) AS reversed
  FROM p
), kept AS (
  SELECT marked.*,
         MAX(CASE WHEN p_dir = 'C' THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref, p_cif)
         + MAX(CASE WHEN p_dir = 'D' THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref, p_cif) AS sides,
         COALESCE(ABS(SUM(p_amt) OVER (PARTITION BY p_ref, p_cif)), 0) AS net
  FROM marked WHERE reversed = 0
)
SELECT p_cif AS "client_no", p_day AS "day", {category} AS "category",
       SUM(CASE WHEN p_dir = 'C' THEN ABS(p_amt) ELSE 0 END) AS "money_in",
       SUM(CASE WHEN p_dir = 'D' THEN ABS(p_amt) ELSE 0 END) AS "money_out",
       COUNT(p_id) AS "postings"
FROM kept
WHERE p_ref IS NULL OR NOT (sides = 2 AND net < 0.005)
GROUP BY p_cif, p_day, {category}"""


def standard_daily(df):
    """The daily totals as the warehouse returns them, with the project's types: client numbers as text, the day as
    a date, amounts and counts as numbers."""
    out = pd.DataFrame({"client_no": df["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True),
                        "day": pd.to_datetime(df["day"]),
                        "category": df["category"].astype(str),
                        "money_in": pd.to_numeric(df["money_in"]).astype(float).fillna(0.0),
                        "money_out": pd.to_numeric(df["money_out"]).astype(float).fillna(0.0),
                        "postings": pd.to_numeric(df["postings"]).astype("int64")})
    return out


def set_cutoff():
    """The forecast's cut-off: the last day of data, by default yesterday. The project variable cutoff_override
    (YYYY-MM-DD) fixes it instead: the sandbox's simulator sets it, and it reruns a past day. Sets the project
    variables cutoff_date, as_of_date and load_id (the data's as-of date), which the recipes read. The staging recipe
    stops if the warehouse has no posting up to the cut-off day.
    Run by the first step of each scenario: from warehouse import set_cutoff; set_cutoff()"""
    import dataiku
    project = dataiku.api_client().get_default_project()
    variables = project.get_variables()
    override = variables["standard"].get("cutoff_override")
    cutoff = pd.Timestamp(override) if override else pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    variables["standard"].update(cutoff_date=f"{cutoff:%Y-%m-%d}", as_of_date=f"{cutoff:%Y-%m-%d}",
                                 load_id=f"EDW_{cutoff:%Y%m%d}")
    project.set_variables(variables)
    print(f"Cut-off day {cutoff:%Y-%m-%d}" + (" (cutoff_override)" if override else " (yesterday)"))
    return f"{cutoff:%Y-%m-%d}"


def check_warehouse_loaded():
    """MOCK. Second step of each scenario: stops the scenario if the warehouse's load of the cut-off day is not
    finished (the ETL from the source systems into EDW). It always passes for now. Replace its body with the bank's
    real check, for example a read-only query on the warehouse's load-control table:
        SELECT STATUS FROM EDW.<LOAD_CONTROL_TABLE> WHERE LOAD_DATE = DATE '<cutoff_date>'   -- must say finished
    and raise an error if it is not finished. Whatever it says, staging also stops if the warehouse has no posting
    up to the cut-off day. Run as: from warehouse import check_warehouse_loaded; check_warehouse_loaded()"""
    import dataiku
    cutoff = dataiku.api_client().get_default_project().get_variables()["standard"]["cutoff_date"]
    print(f"MOCK warehouse check: the load of {cutoff} is assumed finished. Replace check_warehouse_loaded in "
          "warehouse.py with the bank's real check.")
    return True


def clear_daily_flows():
    """Step of the scenario Full rebuild (after the cut-off and the warehouse check): empties daily_flows and staging_log, so that the
    staging recipe reads the whole history again. Run as: from warehouse import clear_daily_flows; clear_daily_flows()"""
    import dataiku
    from cashflow import clear_daily
    clear_daily()
    dataiku.api_client().get_default_project().get_dataset("staging_log").clear()
    print("daily_flows and staging_log emptied: the staging recipe will read the whole history")


for _table, _columns in COLUMNS.items():          # warehouse names unique within a table
    assert len(set(_columns.values())) == len(_columns), _table
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared by every zone (project library: python/cashflow.py): dates, the business rule on amounts, and the
interfaces between the zones:
    1. the data (staging -> training, lab and inference): daily_flows (one row per client, day and category:
       money_in, money_out, postings, over each client's whole history; a folder of Parquet files, one per group of
       clients and month, see write_daily) and the dataset clients (one row per business client). Every model gets
       these and nothing else. Inference gives each model the last INPUT_DAYS days of each client (the last year),
       laid out by model_rows: one row per client, category and window, one column per day.
    2. the model (lab -> training): models.py.
    3. the saved model (training -> inference): SAVED_MODEL; inference uses its active version.
The forecast: per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 3 windows of 30 days after the cut-off day (the last day of data: yesterday): days
1-30, 31-60 and 61-90, as P10, EV and P90 in SAR."""
import logging
import os
import warnings

import numpy as np
import pandas as pd

try:
    import polars as pl
except ImportError:         # the setup notebooks run on Dataiku's built-in Python: they only use this file's settings
    pl = None

# Quieter job logs, for every recipe (they all import this file): Dataiku's own writer prints, for each block it
# writes, pandas warnings about its own code (DataFrame is highly fragmented, is_datetime64tz_dtype is deprecated),
# and an INFO line every 10 s while it waits for the next chunk (No data to send, waiting more...). Harmless: hidden.
# Our own warnings and Dataiku's real warnings and errors still show.
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning, module=r"dataiku\.")
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"dataiku\.")
logging.getLogger("dataiku.core.dataset_write").setLevel(logging.WARNING)
# The same for each file written to a folder (Initializing folder writer, Pipe to generator thread, Sending data, Got
# end mark, ...: several lines per file, hundreds of files): Dataiku logs them on the root logger, so they are dropped
# by where they come from. Only their INFO lines: warnings and errors still show.
_CHATTY = ("dkuio.py", "write_base.py", "managed_folder.py")
logging.getLogger().addFilter(
    lambda record: record.levelno >= logging.WARNING
    or not record.pathname.replace("\\", "/").endswith(tuple(f"dataiku/core/{f}" for f in _CHATTY)))

PERIOD_DAYS = 30            # the forecast's windows, and the periods the models work with, are 30 days long
HORIZONS = (1, 2, 3)        # the windows forecast: 1 = days 1-30 after the cut-off day, 2 = days 31-60, 3 = 61-90
TOTAL = "TOTAL"             # the category of a client's all-categories net cash flow
SAVED_MODEL = "cash_forecast"
INPUT_DAYS = 365            # what a model gets for each client: its daily flows of the last 365 days up to the
                            # cut-off day (a forecaster can ask for another number: its params["input_days"])
MIN_HISTORY_DAYS = INPUT_DAYS   # a client needs this many days of history up to the cut-off to get a forecast (else
                            # status NOT_ENOUGH_HISTORY in the final forecast)
MIN_TRAINING_MONTHS = 21    # history needed to train and test a model: a year of input, the 3 windows after it, and
                            # the 6 periods of 30 days the test keeps aside (about 635 days). Less is fine to test
                            # staging only
FORECAST_COLUMNS = ["client_no", "category", "window", "p10", "ev", "p90"]   # what a forecaster returns


def _plain(df):
    """Dates as plain (time zone free) timestamps, client numbers as text."""
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    if "client_no" in df:
        df["client_no"] = df["client_no"].astype(str)
    return df


def read_dataset(name, columns=None):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps and client numbers as text."""
    import dataiku
    return _plain(dataiku.Dataset(name).get_dataframe(columns=columns))


DAILY_FLOWS = "daily_flows"     # interface 1: a managed folder of Parquet files, written by staging
DAILY_COLUMNS = ["client_no", "day", "category", "money_in", "money_out", "postings"]
DAILY_SCHEMA = dict(zip(DAILY_COLUMNS, [pl.Utf8, pl.Datetime("us"), pl.Utf8, pl.Float64, pl.Float64, pl.Int64])) if pl \
    else {}
MANIFEST = "/manifest.json"     # in the folder: the current files (which group of clients and month each holds)
CLIENTS_PER_GROUP = 5000        # clients per group when the daily flows are written from nothing


def client_groups(clients, groups):
    """The group of each client number (a Polars Series of numbers 0 .. groups - 1): a client never changes group
    (CRC32 of its number), so its rows are always in the files of the same group."""
    import zlib
    numbers = pl.Series(clients, dtype=pl.Utf8)
    unique = numbers.unique()
    group_of = dict(zip(unique.to_list(), (zlib.crc32(n.encode()) % groups for n in unique.to_list())))
    return numbers.replace_strict(group_of, return_dtype=pl.Int64)


def local_path(folder, path=""):
    """A file of a Dataiku folder, on this server's disk: read straight from there, not downloaded through Dataiku's
    server (one HTTP request per file). Every folder of the project is on the local filesystem."""
    if getattr(folder, "local_root", None) is None:
        try:
            folder.local_root = folder.get_path()
        except Exception as e:
            raise RuntimeError(f"The folder {folder.short_name} is not on the Dataiku server's own disk: its connection "
                               f"must be a local filesystem one (it is read straight from disk). {e}") from None
    return os.path.join(folder.local_root, path.lstrip("/"))


def read_manifest(name=DAILY_FLOWS, prefix="", **kwargs):
    """The folder and its manifest (None when nothing is written yet): {"groups": number of groups, "files":
    {group: {month: [path, rows]}}, "first_days": path, "rows": rows in all}. prefix: a subfolder holding the daily
    flows ("" in production; the load test keeps one per size of fake data). kwargs: for dataiku.Folder."""
    import json
    import dataiku
    folder = dataiku.Folder(name, **kwargs)
    path = local_path(folder, prefix + MANIFEST)
    if not os.path.exists(path):
        return folder, None
    with open(path) as f:
        return folder, json.load(f)


def read_parquet(folder, path, columns=None):
    """One Parquet file of a folder, as a Polars DataFrame, read straight from disk (local_path)."""
    return pl.read_parquet(local_path(folder, path), columns=columns)


def read_group(folder, manifest, group, since=None):
    """The daily flows of a group of clients (Polars), from the day `since` on (only the files of its months are
    read), or all of them."""
    months = (manifest["files"].get(str(group)) or {})
    first = None if since is None else f"{since:%Y-%m}"
    parts = [read_parquet(folder, path) for month, (path, _) in sorted(months.items()) if first is None or month >= first]
    daily = pl.concat(parts) if parts else pl.DataFrame(schema=DAILY_SCHEMA)
    return daily if since is None else daily.filter(pl.col("day") >= since)


def read_first_days(folder, manifest):
    """Each client's first posting day: client_no, first_day (Polars)."""
    return read_parquet(folder, manifest["first_days"])


def read_daily(name=DAILY_FLOWS, columns=None, **kwargs):
    """All of daily_flows as one pandas DataFrame (for training, the lab, the sample data; the nightly forecast reads
    one group of clients at a time, read_group)."""
    folder, manifest = read_manifest(name, **kwargs)
    files = [path for months in (manifest or {}).get("files", {}).values() for path, _ in months.values()]
    daily = pl.concat([read_parquet(folder, p, columns) for p in files]) if files else pl.DataFrame(schema=DAILY_SCHEMA)
    return _plain(daily.to_pandas())


def write_daily(parts, groups, name=DAILY_FLOWS, base=None, prefix=""):
    """Write daily flows: each part (pandas or Polars, DAILY_COLUMNS) holds rows of some groups of clients; its rows of
    each group and calendar month go to one Parquet file, group_0007/2026-11_<run>.parquet, added to the rows already
    in that file (base: the current manifest, when adding new days; None: from nothing). Files are never changed in
    place: new ones are written, then the manifest points to them (a failed run leaves the daily flows as they
    were), then the files it no longer lists are removed. Also each client's first posting day (first_days). prefix:
    a subfolder (see read_manifest), the only place files are written and removed. Returns the new manifest."""
    import io
    import json
    import uuid
    import dataiku
    folder = dataiku.Folder(name)
    run = f"{pd.Timestamp.now():%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"
    files = {g: dict(months) for g, months in (base or {}).get("files", {}).items()}
    firsts = [read_first_days(folder, base)] if base else []

    def write(path, frame):
        buffer = io.BytesIO()
        frame.write_parquet(buffer)
        with folder.get_writer(path) as writer:
            writer.write(buffer.getvalue())

    for part in parts:
        part = (part if isinstance(part, pl.DataFrame) else pl.from_pandas(part[DAILY_COLUMNS])).select(
            [pl.col(c).cast(k) for c, k in DAILY_SCHEMA.items()])
        if part.is_empty():
            continue
        firsts.append(part.group_by("client_no").agg(first_day=pl.col("day").min()))
        part = part.with_columns(group=client_groups(part["client_no"], groups), month=pl.col("day").dt.strftime("%Y-%m"))
        for (group, month), rows in part.group_by(["group", "month"]):
            group, rows = str(group), rows.drop(["group", "month"])
            before = files.get(group, {}).get(month)
            if before:
                rows = pl.concat([read_parquet(folder, before[0]), rows])
            path = f"{prefix}/group_{int(group):04d}/{month}_{run}.parquet"
            write(path, rows.sort(["client_no", "day", "category"]))
            files.setdefault(group, {})[month] = [path, len(rows)]
    first_days = f"{prefix}/first_days_{run}.parquet"
    write(first_days, pl.concat(firsts).group_by("client_no").agg(pl.col("first_day").min()).sort("client_no")
          if firsts else pl.DataFrame(schema={"client_no": pl.Utf8, "first_day": pl.Datetime("us")}))
    manifest = {"groups": groups, "files": files, "first_days": first_days,
                "rows": sum(rows for months in files.values() for _, rows in months.values())}
    with folder.get_writer(prefix + MANIFEST) as writer:
        writer.write(json.dumps(manifest).encode())
    kept = {first_days, prefix + MANIFEST} | {path for months in files.values() for path, _ in months.values()}
    for path in folder.list_paths_in_partition():
        if path.startswith(prefix + "/") and path not in kept:
            folder.delete_path(path)
    return manifest


def clear_daily(name=DAILY_FLOWS):
    """Remove every file of daily_flows (the scenario Full rebuild)."""
    import dataiku
    folder = dataiku.Folder(name)
    for path in folder.list_paths_in_partition():
        folder.delete_path(path)


def peak_memory_mb():
    """The most memory this process used, MB (Linux: VmHWM)."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmHWM:"):
                    return round(int(line.split()[1]) / 1024)
    except OSError:
        pass
    import resource
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024)


def append_row(name, columns, row):
    """Add one row (or a list of rows, written together: a recipe writes its output once) to a log dataset (an appended output of the recipe) with fixed column types, so that rows of
    failed runs (with empty values) never change how the earlier rows are read. columns: {name: Dataiku type}, one of
    string, bigint, double, boolean. When the columns changed since the last row (a new version of the code), the
    earlier rows are first read with the old columns and written again with the new ones, by name (a new column is
    empty in them), so they never end up under the wrong column."""
    import dataiku
    kinds = {"string": "string", "bigint": "Int64", "double": "float64", "boolean": "boolean"}
    frame = pd.DataFrame([{c: r.get(c) for c in columns} for r in (row if isinstance(row, list) else [row])])
    dataset = dataiku.Dataset(name)
    before = [c["name"] for c in dataset.read_schema(raise_if_empty=False)]
    if before and before != list(columns):
        try:
            earlier = dataiku.Dataset(name, ignore_flow=True).get_dataframe()  # read with the old columns
        except Exception:                                   # its data was cleared (only the old columns are left)
            earlier = pd.DataFrame(columns=before)
        print(f"{name}: its columns changed; its {len(earlier)} earlier rows are written again with the new columns")
        frame = pd.concat([earlier.reindex(columns=list(columns)), frame], ignore_index=True)
        dataset.spec_item["appendMode"] = False                                 # this time: all the rows again
    for c, kind in columns.items():
        frame[c] = (frame[c].where(frame[c].notna(), "").astype(str) if kind == "string"
                    else pd.to_numeric(frame[c], errors="coerce").round().astype(kinds[kind]) if kind == "bigint"
                    else pd.to_numeric(frame[c], errors="coerce").astype(kinds[kind]) if kind == "double"
                    else frame[c].astype("boolean"))
    dataset.write_schema([{"name": c, "type": kind} for c, kind in columns.items()])
    with dataset.get_writer() as writer:
        writer.write_dataframe(frame)


class Progress:
    """Progress lines in the job's log, for long steps: how much is done (of how much, when known), the percentage,
    the time so far and about how long is left. One line at most every `every` seconds, and the last one always. They
    start with PROGRESS, so the setup notebook can show them while it waits for the job.
        progress = Progress("reading postings", total=41_000_000); progress.add(200_000) ... progress.finish()"""

    def __init__(self, what, total=None, every=30):
        import time
        self.what, self.total, self.every = what, total, every
        self.started = self.printed = time.time()
        self.done = 0
        print(f"PROGRESS {what}: starting" + (f", {total:,} to do" if total else ""), flush=True)

    def add(self, n=1):
        import time
        self.done += n
        if time.time() - self.printed >= self.every:
            self.show()

    def finish(self):
        self.show(last=True)

    def show(self, last=False):
        import time
        now = time.time()
        self.printed = now
        minutes = (now - self.started) / 60
        if last:
            print(f"PROGRESS {self.what}: done, {self.done:,} in {minutes:.1f} min", flush=True)
            return
        line = f"PROGRESS {self.what}: {self.done:,}"
        if self.total:
            share = min(self.done / self.total, 1.0)
            line += f" of {self.total:,} ({share:.0%})"
        line += f", {minutes:.1f} min so far"
        if self.total and 0 < share < 1:
            line += f", about {minutes * (1 - share) / share:.0f} min left"
        print(line, flush=True)


def waiting(what, every=60):
    """While the warehouse runs a query and no row has come back yet, a PROGRESS line every `every` seconds, so a
    long query never looks stuck. Returns an Event: set it when the first row arrives (or the query ends)."""
    import threading
    import time
    stop, started = threading.Event(), time.time()

    def beat():
        while not stop.wait(every):
            print(f"PROGRESS {what}: no row yet, the warehouse is still running the query "
                  f"({(time.time() - started) / 60:.0f} min)", flush=True)

    threading.Thread(target=beat, daemon=True).start()
    return stop


class Spread:
    """Rows spread over n files on local disk by the hash of a column, then read back one file at a time: all the
    rows with the same value of that column end up in the same file, and memory holds one file, not the whole table.
    Used by staging (postings by reference, then daily flows by client)."""

    def __init__(self, folder, n):
        import os
        os.makedirs(folder, exist_ok=True)
        self.n = max(1, int(n))
        self.paths = [os.path.join(folder, f"part_{i}.pkl") for i in range(self.n)]

    def add(self, df, key):
        """key: a column name, or a Series (same index as df) to spread by."""
        import pickle
        key = df[key] if isinstance(key, str) else key
        part = pd.util.hash_pandas_object(key.astype(str), index=False).to_numpy() % self.n
        for i, rows in df.groupby(part):
            with open(self.paths[i], "ab") as f:
                pickle.dump(rows, f, protocol=4)

    def parts(self):
        """Each file's rows (a DataFrame); files that got no rows are skipped."""
        import os
        import pickle
        for path in self.paths:
            if not os.path.exists(path):
                continue
            frames = []
            with open(path, "rb") as f:
                while True:
                    try:
                        frames.append(pickle.load(f))
                    except EOFError:
                        break
            yield pd.concat(frames, ignore_index=True)


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def cutoff_date():
    """The forecast's cut-off: the last day of data, by default yesterday (project variable cutoff_date, set by the
    first step of each scenario: warehouse.set_cutoff)."""
    import dataiku
    return pd.Timestamp(dataiku.get_custom_variables()["cutoff_date"])


def whole_number(name, value):
    """A setting that is a whole number, or None for "all" (empty, None, -1 or "all"). Stops with a clear message
    otherwise."""
    value = "" if value is None else str(value).strip()
    if value.lower() in ("", "none", "all", "-1"):
        return None
    try:
        number = float(value)
    except ValueError:
        number = 0.5
    if number != int(number) or number < 1:
        raise ValueError(f"{name} is {value!r}: it must be a whole number (1 or more), or -1 / None for all")
    return int(number)


def history_start(cutoff):
    """The first day staging reads when it reads everything (the first run, or the scenario Full rebuild): the
    project variable history_months months before the cut-off day, or None for the whole history. With less than
    about 18 months, few clients reach the history a forecast and the model test need (MIN_HISTORY_DAYS, and the
    last 6 periods of 30 days kept aside): fine to test the pipeline, not to judge a model."""
    import dataiku
    months = whole_number("history_months", dataiku.get_custom_variables().get("history_months"))
    return None if months is None else cutoff - pd.DateOffset(months=months) + pd.Timedelta(days=1)


def clients_limit():
    """How many business clients staging reads (project variable clients_limit): None for all of them."""
    import dataiku
    return whole_number("clients_limit", dataiku.get_custom_variables().get("clients_limit"))


def until(daily, cutoff):
    """The daily flows known at the cut-off: the days up to the cut-off day (included)."""
    return daily[pd.to_datetime(daily["day"]) <= cutoff]


def window(cutoff, h):
    """The first and last day of window h after the cut-off day (h = 1: days 1-30)."""
    return cutoff + pd.Timedelta(days=PERIOD_DAYS * (h - 1) + 1), cutoff + pd.Timedelta(days=PERIOD_DAYS * h)


def finish(ev, p10, p90, index=None):
    """P10 <= EV <= P90 (SAR). The net cash flow can be below zero."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    return pd.DataFrame({"p10": np.minimum(p10, ev), "ev": ev, "p90": np.maximum(p90, ev)}, index=index).round(2)


def _sums(d, key):
    """Net cash flow per client, category (and TOTAL) and key."""
    by_category = d.groupby(["client_no", "category", key])["net"].sum()
    total = d.groupby(["client_no", key])["net"].sum().to_frame().assign(category=TOTAL)
    total = total.set_index("category", append=True).reorder_levels(["client_no", "category", key])["net"]
    return pd.concat([by_category, total])


def _wide(sums, columns):
    """One row per client and category, the given columns; 0 after a series' first value, empty before it."""
    if sums.empty:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=["client_no", "category"]))
    wide = sums.unstack(sums.index.names[-1]).reindex(columns=columns)
    started = wide.notna().cumsum(axis=1) > 0
    return wide.fillna(0.0).where(started).sort_index()


def period_matrix(daily, cutoff):
    """The net cash flow (money in minus money out) per 30-day period ending at the cut-off day: one row per client
    and category (and per client in TOTAL), one column per period, named by its last day (the last column is the 30
    days up to the cut-off, the one before the 30 days before, ...). 0 for a period with no posting after the
    series' first period, empty before it. The window forecast h is the period ending cutoff + 30 x h days."""
    d = until(daily, cutoff)
    back = (cutoff - pd.to_datetime(d["day"])).dt.days // PERIOD_DAYS
    d = d.assign(period=(cutoff - pd.to_timedelta(back * PERIOD_DAYS, unit="D")).values,
                 net=(d["money_in"] - d["money_out"]).values)
    if d.empty:
        return _wide(pd.Series(dtype=float), [])
    oldest = int(back.max())
    return _wide(_sums(d, "period"), [cutoff - pd.Timedelta(days=PERIOD_DAYS * k) for k in range(oldest, -1, -1)])


def period_series(daily, cutoff):
    """period_matrix as rows: client_no, category, period (its last day), net (no row before a series' first)."""
    long = period_matrix(daily, cutoff).reset_index().melt(id_vars=["client_no", "category"], var_name="period",
                                                           value_name="net")
    long["period"] = pd.to_datetime(long["period"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "period"]).reset_index(drop=True)


def monthly_series(daily, cutoff):
    """For the EDA: the net cash flow per client, category (and TOTAL) and calendar month, up to the cut-off day (its
    month is partial): client_no, category, month, net (no row before a series' first month)."""
    d = until(daily, cutoff)
    d = d.assign(month=month_start(d["day"]).values, net=(d["money_in"] - d["money_out"]).values)
    if d.empty:
        return pd.DataFrame(columns=["client_no", "category", "month", "net"])
    months = list(pd.date_range(d["month"].min(), month_start([cutoff])[0], freq="MS"))
    long = _wide(_sums(d, "month"), months).reset_index().melt(id_vars=["client_no", "category"], var_name="month",
                                                                value_name="net")
    long["month"] = pd.to_datetime(long["month"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "month"]).reset_index(drop=True)


def input_days(model=None):
    """How many days of daily flows a model gets for each client: its params["input_days"], else INPUT_DAYS."""
    return int((getattr(model, "params", None) or {}).get("input_days") or INPUT_DAYS)


def day_columns(days=INPUT_DAYS):
    """The names of the model's day columns, the oldest first: day_365, ..., day_1 (day_1: the cut-off day)."""
    return [f"day_{days - i}" for i in range(days)]


def day_matrix(daily, cutoff, days=INPUT_DAYS):
    """The net cash flow (money in minus money out) of each of the last `days` days up to the cut-off day, per client
    and category (and per client in TOTAL: all its categories together); 0 for a day with no posting. Returns the
    keys (Polars: client_no, category, sorted) and the matrix (numpy float32, one row per key, the oldest day first).
    daily: Polars or pandas."""
    d = daily if isinstance(daily, pl.DataFrame) else pl.from_pandas(daily[DAILY_COLUMNS])
    end = pd.Timestamp(cutoff).to_pydatetime()
    d = d.filter(pl.col("day").is_between(end - pd.Timedelta(days=days - 1), end)).select(
        pl.col("client_no").cast(pl.Utf8), pl.col("category").cast(pl.Utf8),
        (days - 1 - (pl.lit(end) - pl.col("day")).dt.total_days()).alias("column"),
        (pl.col("money_in") - pl.col("money_out")).alias("net"))
    d = pl.concat([d, d.with_columns(category=pl.lit(TOTAL))])
    keys = d.select("client_no", "category").unique().sort("client_no", "category")
    d = d.join(keys.with_row_index("series"), on=["client_no", "category"])
    flat = d["series"].to_numpy().astype(np.int64) * days + d["column"].to_numpy()
    matrix = np.bincount(flat, weights=d["net"].to_numpy(), minlength=len(keys) * days)
    return keys, matrix.reshape(len(keys), days).astype(np.float32)


def model_rows(keys, matrix):
    """The model's input (interface 2): one row per client, category and window (1, 2, 3: days 1-30, 31-60, 61-90
    after the cut-off day), its columns client_no, category, window, day_365 ... day_1 (the net cash flow of each
    day, SAR). keys, matrix: from day_matrix. Polars."""
    days = pl.DataFrame(matrix, schema=day_columns(matrix.shape[1]), orient="row")
    return pl.concat([keys.with_columns(window=pl.lit(h, pl.Int8)).hstack(days) for h in HORIZONS])
'''

In [ ]:
MODELS_CODE = r'''
"""Models (project library: python/models.py): interface 2, the model, the placeholder models and the test every
model goes through. Training (train_and_test, champion_challenger) and the lab use it; the nightly forecast only gives
the production model its input rows (forecast.py).

A model (a "forecaster" in the project variable that picks it) is a class with:
    name            its name. The project variable forecaster says which one the scenario Retrain, test and promote
                    trains (default: daily_catboost).
    params          its settings (a dict), recorded in Experiment Tracking. params["input_days"]: how many days of
                    each client it gets (default 365).
    fit(daily, clients, cutoff)
                    learns from the data known at the cut-off day (interface 1, cashflow.py): daily: daily_flows
                    (client_no, day, category, money_in, money_out, postings, pandas), every day up to the cut-off
                    day; clients: the clients dataset; cutoff: the cut-off day (a Timestamp: the last day of data).
    predict_rows(rows)
                    what the nightly forecast calls. rows: a Polars DataFrame (cashflow.model_rows), one row per
                    client, category (each category the client has, and TOTAL: all its categories together) and window
                    (1 = days 1-30 after the cut-off day, 2 = days 31-60, 3 = days 61-90): client_no, category, window,
                    day_365 ... day_1 (the net cash flow of each of the last input_days days, day_1 the cut-off day).
                    Returns a numpy array, one row per input row: P10, EV, P90 of the window's net cash flow (SAR).
    predict(daily, clients, cutoff)
                    the same from daily flows, for the test, champion / challenger and the lab: DayModel gives it.
The data scientist's own models go in forecasters.py: the setup notebook creates that file once and never overwrites
it. lab_compare (notebook forecast_lab) tries them against the production model without touching production.
"""
import numpy as np
import pandas as pd
import polars as pl

from cashflow import (FORECAST_COLUMNS, HORIZONS, INPUT_DAYS, PERIOD_DAYS, cutoff_date, day_matrix, finish,
                      model_rows, period_matrix, period_series, until)

HOLDOUT = 6                 # the last 6 periods of 30 days are kept aside to test the models
OUTPUT = ["p10", "ev", "p90"]


class DayModel:
    """Base of the placeholders: predict (from daily flows) through predict_rows (from the model's input rows), so the
    test and the nightly forecast run the same code."""
    params = {"input_days": INPUT_DAYS}

    def predict(self, daily, clients, cutoff):
        rows = model_rows(*day_matrix(until(daily, cutoff), cutoff, self.params["input_days"]))
        if rows.is_empty():
            return pd.DataFrame(columns=FORECAST_COLUMNS)
        values = self.predict_rows(rows)
        out = rows.select("client_no", "category", "window").to_pandas()
        out["window"] = out["window"].astype(int)
        out[OUTPUT] = values
        return out[FORECAST_COLUMNS]

    @staticmethod
    def days(rows):
        """The day columns of the input rows, as a numpy float32 array (the oldest day first)."""
        return rows.select(pl.col("^day_.*$")).to_numpy().astype(np.float32, copy=False)


class RecentAverage(DayModel):
    """A simple model with no machine learning, to compare others with (and fall back on). Per client and category,
    EV = the average net cash flow of the last 3 periods of 30 days (the same for the 3 windows). P10 and P90 = EV -/+
    width x the series' standard deviation over the last 12 periods, where fit sets width so that, over the 12
    periods before the cut-off, 80% of the actual periods fell inside."""
    name = "recent_average"
    params = {"input_days": INPUT_DAYS, "periods": 3, "spread_periods": 12, "coverage": 0.8}

    def fit(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        n, s = self.params["periods"], self.params["spread_periods"]
        z = []
        for k in range(1, min(12, wide.shape[1] - 1) + 1):         # each of the last 12 periods, from the ones before
            past, actual = wide.iloc[:, :-k], wide.iloc[:, -k]
            ev, sd = past.iloc[:, -n:].mean(axis=1), past.iloc[:, -s:].std(axis=1)
            ok = actual.notna() & ev.notna() & (sd > 0)
            z.append(((actual - ev).abs() / sd)[ok])
        z = pd.concat(z) if z else pd.Series(dtype=float)
        self.width = float(z.quantile(self.params["coverage"])) if len(z) else 1.28
        return self

    def predict_rows(self, rows):
        s = self.params["spread_periods"]
        days = self.days(rows)[:, -PERIOD_DAYS * s:]
        periods = days.reshape(len(days), s, PERIOD_DAYS).sum(axis=2)     # the last 12 periods of 30 days
        ev = periods[:, -self.params["periods"]:].mean(axis=1)
        sd = periods.std(axis=1, ddof=1)
        return np.column_stack([ev - self.width * sd, ev, ev + self.width * sd])


class DailyCatBoost(DayModel):
    """The production placeholder (option A), until the data scientist's models replace it: ONE CatBoost model with
    the MultiQuantile loss gives P10, EV (the median, far less thrown off by a few huge days than the mean) and P90
    together, for every category, TOTAL and the 3 windows. Its inputs are the rows of cashflow.model_rows: the 365
    days (divided by the series' scale so small and large clients look alike), the category (as a number) and the
    window. Nothing else: no sums over 30 days, no calendar.
    Target in training: the net cash flow of the window (the sum of its 30 days), divided by the same scale. Trained on
    rolling cut-offs: every 30 days back from the cut-off (up to `cutoffs` of them), each series whose whole year of
    input is known, each window whose 30 days are known; at most max_clients clients and max_rows rows (random
    samples beyond that, the same every run). threads: CatBoost's threads when predicting (the nightly forecast sets
    it from its workers)."""
    name = "daily_catboost"
    params = {"input_days": INPUT_DAYS, "cutoffs": 24, "max_clients": 20_000, "max_rows": 200_000,
              "iterations": 300, "depth": 6, "learning_rate": 0.1}
    QUANTILES = (0.1, 0.5, 0.9)          # the model's 3 outputs, in this order: P10, EV, P90
    threads = -1

    def new_model(self):
        from catboost import CatBoostRegressor
        alphas = ",".join(str(q) for q in self.QUANTILES)
        return CatBoostRegressor(loss_function=f"MultiQuantile:alpha={alphas}", iterations=self.params["iterations"],
                                 depth=self.params["depth"], learning_rate=self.params["learning_rate"],
                                 random_seed=0, verbose=False, allow_writing_files=False)

    def _inputs(self, days, codes, window):
        """The model's inputs, and each row's scale."""
        scale = np.maximum(np.abs(days).mean(axis=1) * PERIOD_DAYS, 100.0).astype(np.float32)
        return np.column_stack([days / scale[:, None], codes, window]).astype(np.float32), scale

    def _codes(self, categories):
        """Each category as a number (its place among the categories seen in training; empty if new)."""
        code = {c: i for i, c in enumerate(self.categories)}
        return pl.Series(categories, dtype=pl.Utf8).replace_strict(code, default=None, return_dtype=pl.Float32)\
            .fill_null(np.nan).to_numpy()

    def fit(self, daily, clients, cutoff):
        days, rng = self.params["input_days"], np.random.default_rng(0)
        known = until(daily, cutoff)
        if known.empty:
            raise ValueError(f"{self.name}: no daily flows up to {cutoff:%Y-%m-%d} to train on")
        numbers = known["client_no"].astype(str).unique()
        if len(numbers) > self.params["max_clients"]:
            numbers = rng.choice(np.sort(numbers), self.params["max_clients"], replace=False)
            known = known[known["client_no"].astype(str).isin(set(numbers))]
        first = pd.to_datetime(known.groupby("client_no")["day"].min())
        span = (cutoff - first.min()).days + 1
        keys, matrix = day_matrix(known, cutoff, span)                   # every day, the oldest first
        self.categories = sorted(set(keys["category"]))
        codes = self._codes(keys["category"])
        start = span - 1 - (cutoff - first.reindex(keys["client_no"].to_list())).dt.days.to_numpy()
        blocks = []                      # (the cut-off's column, window, the series whose input year is known)
        for k in range(1, self.params["cutoffs"] + 1):
            end = span - 1 - PERIOD_DAYS * k
            if end - days + 1 < 0:
                break
            rows = np.flatnonzero(start <= end - days + 1)
            blocks += [(end, h, rows) for h in HORIZONS if k >= h and len(rows)]
        total = sum(len(rows) for _, _, rows in blocks)
        if not total:
            raise ValueError(f"{self.name}: not enough history to train: a series needs {days} days of input and the "
                             f"{PERIOD_DAYS} days after them (daily flows from {first.min():%Y-%m-%d} to "
                             f"{cutoff:%Y-%m-%d})")
        keep = min(1.0, self.params["max_rows"] / total)
        Xs, ys = [], []
        for end, h, rows in blocks:
            rows = rows[rng.random(len(rows)) < keep]
            X, scale = self._inputs(matrix[rows, end - days + 1:end + 1], codes[rows], np.full(len(rows), h))
            Xs.append(X)
            ys.append(matrix[rows, end + PERIOD_DAYS * (h - 1) + 1:end + PERIOD_DAYS * h + 1].sum(axis=1) / scale)
        self.model = self.new_model().fit(np.vstack(Xs), np.concatenate(ys))
        return self

    def predict_rows(self, rows):
        X, scale = self._inputs(self.days(rows), self._codes(rows["category"]), rows["window"].to_numpy())
        return self.model.predict(X, thread_count=self.threads) * scale[:, None]


def known_forecasters():
    """The forecasters Retrain can train, by name: the placeholders (daily_catboost, recent_average) and those
    listed in FORECASTERS in forecasters.py. Also returns the problem reading forecasters.py, if any (the placeholders
    still work)."""
    found, problem = {DailyCatBoost.name: DailyCatBoost, RecentAverage.name: RecentAverage}, None
    try:
        import forecasters
        for forecaster in forecasters.FORECASTERS:
            found[forecaster.name] = forecaster
    except Exception as e:
        problem = f"forecasters.py could not be read ({type(e).__name__}: {e})"
    return found, problem


def new_forecaster(name=None):
    """A new, untrained forecaster: the one named by the project variable forecaster (default daily_catboost)."""
    import dataiku
    name = name or dataiku.get_custom_variables().get("forecaster") or DailyCatBoost.name
    found, problem = known_forecasters()
    if name not in found:
        raise ValueError(f"No forecaster named {name!r} (project variable forecaster). Known: {', '.join(found)}. "
                         + (problem if problem else "Add yours to FORECASTERS in forecasters.py."))
    forecaster = found[name]()
    missing = [a for a in ("name", "fit", "predict") if not hasattr(forecaster, a)]
    if missing:
        raise ValueError(f"Forecaster {name} has no {', '.join(missing)} (see models.py)")
    return forecaster


def forecast(forecaster, daily, clients, cutoff):
    """P10, EV and P90 of a trained forecaster (SAR), checked, with P10 <= EV <= P90: one row per client, category and
    window. Everything that predicts goes through here: the test, champion / challenger, the lab and the
    nightly forecast."""
    out = forecaster.predict(daily, clients, cutoff)
    name = getattr(forecaster, "name", type(forecaster).__name__)
    if not isinstance(out, pd.DataFrame) or not set(FORECAST_COLUMNS) <= set(out.columns):
        raise ValueError(f"Forecaster {name}: predict must return a DataFrame with the columns "
                         f"{', '.join(FORECAST_COLUMNS)}")
    empty = [c for c in OUTPUT if out[c].isna().any()]
    if empty:
        raise ValueError(f"Forecaster {name}: predict returned empty values in {', '.join(empty)}")
    out = out[FORECAST_COLUMNS].reset_index(drop=True)
    out["client_no"] = out["client_no"].astype(str)
    out["window"] = out["window"].astype(int)
    out[OUTPUT] = finish(out["ev"], out["p10"], out["p90"], out.index)
    return out.sort_values(["client_no", "category", "window"]).reset_index(drop=True)


def active(daily, clients):
    """The active clients and their daily flows: the clients the test and the forecast are made for."""
    clients = clients[clients["status"] == "ACTIVE"]
    return daily[daily["client_no"].isin(set(clients["client_no"]))], clients


def test_forecasts(fitted, daily, clients, cutoff):
    """The test, the same for every model: a forecaster trained without the last 6 periods of 30 days forecasts the
    windows after each test cut-off (the 6 days, 30 days apart, before the cut-off) from the data known at that day,
    for the active clients. Returns one row per client, category, test cut-off and window whose days are known:
    target (the actual net cash flow), p10, ev, p90 and baseline (the simple guess: the same as the 30 days before the
    test cut-off)."""
    daily, clients = active(daily, clients)
    known = period_series(daily, cutoff)
    actual = known.rename(columns={"period": "target_end", "net": "target"})
    last = known.rename(columns={"period": "test_cutoff", "net": "baseline"})
    parts = []
    for k in range(HOLDOUT):
        t = cutoff - pd.Timedelta(days=PERIOD_DAYS * (HOLDOUT - k))
        f = forecast(fitted, until(daily, t), clients, t).assign(test_cutoff=t)
        f["target_end"] = f["window"].map({h: t + pd.Timedelta(days=PERIOD_DAYS * h) for h in HORIZONS})
        parts.append(f[f["target_end"] <= cutoff])
    keys = ["client_no", "category"]
    tested = (pd.concat(parts, ignore_index=True)
              .merge(actual, on=keys + ["target_end"], how="left")
              .merge(last, on=keys + ["test_cutoff"], how="left"))
    tested[["target", "baseline"]] = tested[["target", "baseline"]].fillna(0.0)
    return tested


def backtest(forecaster, daily, clients, cutoff):
    """Train the forecaster without the last 6 periods of 30 days (the data known at the first test cut-off), then
    test it on them (test_forecasts). Returns the test rows; the forecaster is now the backtest forecaster."""
    first = cutoff - pd.Timedelta(days=PERIOD_DAYS * HOLDOUT)
    forecaster.fit(until(daily, first), clients, first)
    return test_forecasts(forecaster, daily, clients, cutoff)


def scores(d):
    """Test scores of predictions (columns target, p10, ev, p90, baseline): EV error (WAPE) of the model and of the
    simple guess, and the share of actual values below P10, between P10 and P90, and above P90."""
    size = d["target"].abs().sum()
    wape = lambda column: round((d["target"] - d[column]).abs().sum() / size, 4) if size else np.nan
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0), "wape_model": wape("ev"),
                      "wape_baseline": wape("baseline") if "baseline" in d else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


def test_results(tested):
    """Test scores per window (1, 2, 3) and overall (window ALL), for the TOTAL rows (level TOTAL) and the
    category rows (level CATEGORIES). The overall scores, which promotion uses, are on all rows."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    level = np.where(tested["category"] == "TOTAL", "TOTAL", "CATEGORIES")
    results = pd.concat([
        tested.assign(level=level).groupby(["level", "window"])[scored].apply(scores).reset_index(),
        tested.assign(level=level).groupby("level")[scored].apply(scores).reset_index().assign(window="ALL"),
        scores(tested).to_frame().T.assign(level="ALL", window="ALL")], ignore_index=True)
    results["window"] = results["window"].astype(str)
    return results[["level", "window"] + [c for c in results if c not in ("level", "window")]]


def run_metrics(results):
    """The scores recorded in Experiment Tracking: overall, and the EV error of the TOTAL rows per window.
    results: from test_results."""
    overall = results[(results["level"] == "ALL")].iloc[0]
    metrics = {k: overall[c] for k, c in (("wape", "wape_model"), ("wape_baseline", "wape_baseline"),
                                          ("coverage_p10_p90", "coverage_p10_p90"), ("below_p10", "below_p10"),
                                          ("above_p90", "above_p90"))}
    for row in results[(results["level"] == "TOTAL") & (results["window"] != "ALL")].itertuples():
        metrics[f"wape_total_w{row.window}"] = row.wape_model
    return metrics


def _lab_test():
    """The lab's data and the production model's test, the same as champion / challenger's: daily flows, clients,
    the production model's ID and its backtest forecaster's test rows (None without a production model)."""
    import tracking
    from cashflow import read_daily, read_dataset
    daily, clients = read_daily(), read_dataset("clients")
    production_id = tracking.production_model_id()
    production = (test_forecasts(tracking.load_backtest(production_id), daily, clients, cutoff_date())
                  if production_id else None)
    return daily, clients, production_id, production


def _by_window(tested):
    """Overall test scores per window (all rows), indexed by window (1, 2, 3, ALL)."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    per = tested.groupby("window")[scored].apply(scores)
    per.index = per.index.astype(str)
    per.loc["ALL"] = scores(tested)
    return per


def _record_try(forecaster, tested, production_id, daily, clients, purpose):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab), with its fingerprint (tracking.describe).
    Returns the run ID."""
    import tracking
    params = {"test_rows": len(tested), "compared_with": production_id or "none"}
    about = tracking.describe(forecaster, daily, clients, cutoff_date(), tested, "lab notebook (lab_compare)")
    run_name = f"{forecaster.name}_{pd.Timestamp.now():%Y%m%dT%H%M}"
    return tracking.log_lab_run(run_name, params, run_metrics(test_results(tested)), forecaster, about, purpose)


def lab_compare(forecasters, record=True, purpose=""):
    """Try several forecasters at once against the production model, on the same test as champion / challenger,
    without touching production. Each forecaster is trained without the last 6 periods of 30 days; each try is
    recorded in Experiment Tracking (experiment cashflow_lab) unless record is False, with `purpose` (what the try
    is for) as a tag. Returns a dict:
        board       one row per model (production first, simple guess last): EV error, P10-P90 coverage, share below
                    P10 and above P90, the decision champion / challenger would take and its reason, the lab run ID
        errors      EV error per window: the simple guess, the production model and each forecaster
        coverage    P10-P90 coverage per window: the production model and each forecaster
        tested      the test rows with each model's p10, ev, p90, by name, for your own analysis"""
    import tracking
    names = [f.name for f in forecasters]
    if len(set(names)) != len(names) or {"production", "simple_guess_baseline"} & set(names):
        raise ValueError(f"Give each forecaster its own name: {names}")
    daily, clients, production_id, production = _lab_test()
    tested = {} if production is None else {"production": production}
    for forecaster in forecasters:
        tested[forecaster.name] = backtest(forecaster, daily, clients, cutoff_date())
    run_ids = ({f.name: _record_try(f, tested[f.name], production_id, daily, clients, purpose) for f in forecasters}
               if record else {})
    reference = scores(production) if production is not None else None
    board = []
    for name, rows in tested.items():
        s = scores(rows)
        decision, reason = (("", f"the production model, {production_id}") if name == "production"
                            else tracking.decide(s, reference))
        board.append({"model": name, "ev_error": s["wape_model"], "coverage_p10_p90": s["coverage_p10_p90"],
                      "below_p10": s["below_p10"], "above_p90": s["above_p90"], "decision": decision,
                      "reason": reason, "lab_run_id": run_ids.get(name, "")})
    board.append({"model": "simple_guess_baseline", "ev_error": scores(next(iter(tested.values())))["wape_baseline"],
                  "decision": "", "reason": "the same as the 30 days before: the bar to beat", "lab_run_id": ""})
    board = pd.DataFrame(board).set_index("model")
    per = {name: _by_window(rows) for name, rows in tested.items()}
    first = next(iter(per.values()))
    errors = pd.DataFrame({"simple_guess_baseline": first["wape_baseline"],
                           **{name: found["wape_model"] for name, found in per.items()}})
    coverage = pd.DataFrame({name: found["coverage_p10_p90"] for name, found in per.items()})
    return {"board": board, "errors": errors, "coverage": coverage, "tested": tested}
'''

In [ ]:
FORECAST_CODE = r'''
"""Inference (project library: python/forecast.py): the production model and the nightly forecast. Inference reads
nothing from training except the active version of the saved model cash_forecast (interface 3, cashflow.SAVED_MODEL).
The load test project runs this same file on fake data.

The nightly forecast (forecast_all), Polars from start to end: the groups of clients of daily_flows run in parallel
(`workers` threads: Polars, the Parquet files and CatBoost work outside Python's lock, so threads run side by side;
processes would not be safe here, as Polars and CatBoost keep their own threads), each group on its own from start
to end (forecast_group):
    reading      the files of its last months only, the last input days (365) of its active clients
    preparing    the model's input rows (cashflow.model_rows: client, category, window, day_365 ... day_1)
    model        the model's predict_rows: P10, EV, P90 of each row
    formatting   one row per client (output_rows)
    writing      one Parquet file of the group, straight into the output folder (one group at a time: Dataiku
                 allows one writer per folder)
The output folder holds one subfolder per night, forecast_<cut-off>_<model ID> (kept: the archive), and current.txt,
which points to the newest one only once every group is written: a failed night leaves the previous forecast current.
The SQL table final_forecast is then loaded from it (load_to_sql, or load_to_table: see SQL tables below)."""
import io
import os
import time

import numpy as np
import pandas as pd

try:
    import polars as pl
except ImportError:         # the setup notebooks run on Dataiku's built-in Python: they only use the SQL table helpers
    pl = None

from cashflow import (DAILY_FLOWS, HORIZONS, MIN_HISTORY_DAYS, SAVED_MODEL, TOTAL, Progress, client_groups,
                      day_matrix, input_days, local_path, model_rows, peak_memory_mb, read_first_days, read_group,
                      read_manifest, read_parquet, window)

VALUES = ("p10", "ev", "p90")
STEPS = ("reading", "preparing", "model", "formatting", "writing")
OUTPUT = "forecast"                 # the output folder
CURRENT = "/current.txt"            # in it: the subfolder of the current forecast


def _saved_model_id():
    import dataiku
    project = dataiku.api_client().get_default_project()
    ids = {m["name"]: m["id"] for m in project.list_saved_models()}
    if SAVED_MODEL not in ids:
        raise ValueError(f"Saved model {SAVED_MODEL} not found. Run the setup notebook.")
    return ids[SAVED_MODEL]


def active_model_id():
    """The production model: the active version of the saved model cash_forecast."""
    import dataiku
    model = dataiku.api_client().get_default_project().get_saved_model(_saved_model_id())
    model_id = (model.get_active_version() or {}).get("id")
    if not model_id:
        raise ValueError("No production model yet: run the scenario Retrain, test and promote")
    return model_id


def production_forecaster():
    """The production model (the active version of cash_forecast), and its model ID. The recipe that calls it must
    have the saved model as an input."""
    import dataiku
    model_id = active_model_id()
    predictor = dataiku.Model(_saved_model_id()).get_predictor(model_id)
    return predictor.model.unwrap_python_model().forecaster, model_id


def value_columns():
    """The value columns of the forecast, always the same and in this order: TOTAL first, then each category of
    warehouse.py (CATEGORY_OF_TYPE and OTHER), each for windows 1, 2, 3 (days 1-30, 31-60, 61-90 after the cut-off
    day) and P10, EV, P90: TOTAL_w1_p10, TOTAL_w1_ev, TOTAL_w1_p90, TOTAL_w2_p10, ..., UTILITIES_w3_p90."""
    from warehouse import CATEGORY_OF_TYPE, OTHER
    categories = [TOTAL] + sorted(set(CATEGORY_OF_TYPE.values()) | {OTHER})
    return [f"{c}_w{h}_{v}" for c in categories for h in HORIZONS for v in VALUES]


def output_rows(pred, clients, cutoff):
    """The forecast of a group: ONE row per client of `clients` (client_no, first_day, recent: whether it has a
    posting in its last input days). pred: one row per client, category and window with p10, ev, p90. Columns:
    client_no, cutoff_date (the last day of data), w1_start ... w3_end (the days of each window), status (OK;
    NOT_ENOUGH_HISTORY: less than MIN_HISTORY_DAYS days since its first posting; NO_POSTINGS: none in its last input
    days), history_days, then value_columns() (empty for a category the client does not have, and when not OK)."""
    columns = value_columns()
    wide = (pred.unpivot(index=["client_no", "category", "window"], on=list(VALUES))
            .with_columns(key=pl.format("{}_w{}_{}", "category", "window", "variable"))
            .pivot(on="key", index="client_no", values="value")) if len(pred) else pl.DataFrame(
        schema={"client_no": pl.Utf8})
    unknown = sorted({c.split("_w")[0] for c in wide.columns[1:] if c not in columns})
    if unknown:
        raise ValueError(f"Forecast for categories not in warehouse.py: {', '.join(unknown)}")
    end = cutoff.to_pydatetime().date()
    history = (pl.lit(end) - pl.col("first_day").dt.date()).dt.total_days() + 1
    dates = {"cutoff_date": pl.lit(end)}
    for h in HORIZONS:
        first, last = window(cutoff, h)
        dates[f"w{h}_start"], dates[f"w{h}_end"] = pl.lit(first.date()), pl.lit(last.date())
    out = clients.with_columns(history_days=history.fill_null(0).cast(pl.Int64)).join(wide, on="client_no", how="left")
    status = (pl.when(pl.col("client_no").is_in(wide["client_no"].implode())).then(pl.lit("OK"))
              .when(pl.col("recent") & (pl.col("history_days") < MIN_HISTORY_DAYS)).then(pl.lit("NOT_ENOUGH_HISTORY"))
              .otherwise(pl.lit("NO_POSTINGS")))
    return out.select("client_no", *[v.alias(k) for k, v in dates.items()], status.alias("status"), "history_days",
                      *[pl.col(c).cast(pl.Float64) if c in out.columns else pl.lit(None, pl.Float64).alias(c)
                        for c in columns])


_JOB = {}       # what every group needs, set by forecast_all before the workers start


def forecast_group(group):
    """One group of clients from start to end (see the module's text). Returns its seconds per step and counts."""
    job, times = _JOB, {}
    model, cutoff, days = job["model"], job["cutoff"], job["days"]
    ticked = time.time()
    clients = job["clients"][group]
    since = cutoff - pd.Timedelta(days=days - 1)
    daily = read_group(job["folder"], job["manifest"], group, since).filter(
        pl.col("client_no").is_in(clients["client_no"].implode()))
    clients = clients.with_columns(recent=pl.col("client_no").is_in(daily["client_no"].unique().implode()))
    enough = clients.filter(pl.col("recent") & ((pl.lit(cutoff.to_pydatetime()) - pl.col("first_day")).dt.total_days()
                                                >= MIN_HISTORY_DAYS - 1))
    daily = daily.filter(pl.col("client_no").is_in(enough["client_no"].implode()))
    times["reading"], ticked = time.time() - ticked, time.time()
    rows = model_rows(*day_matrix(daily, cutoff, days))
    times["preparing"], ticked = time.time() - ticked, time.time()
    values = np.asarray(model.predict_rows(rows), dtype=np.float64) if len(rows) else np.zeros((0, 3))
    if values.shape != (len(rows), 3) or not np.isfinite(values).all():
        raise ValueError(f"The model's predict_rows must return one finite P10, EV, P90 per input row: got {values.shape}")
    times["model"], ticked = time.time() - ticked, time.time()
    ev = values[:, 1]
    pred = rows.select("client_no", "category", "window").with_columns(
        p10=np.minimum(values[:, 0], ev).round(2), ev=ev.round(2), p90=np.maximum(values[:, 2], ev).round(2))
    out = output_rows(pred, clients, cutoff).with_columns(**{k: pl.lit(v) for k, v in job["columns"].items()})
    times["formatting"], ticked = time.time() - ticked, time.time()
    buffer = io.BytesIO()
    out.write_parquet(buffer)
    with job["writing"], job["output"].get_writer(f"/{job['version']}/part_{group:05d}.parquet") as writer:
        writer.write(buffer.getvalue())             # one writer at a time: Dataiku allows one per folder
    times["writing"] = time.time() - ticked
    counts = out["status"].value_counts()
    return {**times, "daily_rows": len(daily), "clients": len(out),
            **dict(zip(counts["status"].to_list(), counts["count"].to_list()))}


def forecast_all(model, model_id, clients, cutoff, workers, name=DAILY_FLOWS, output=OUTPUT, columns=None,
                 prefix=""):
    """The nightly forecast of `clients` (Polars or pandas, client_no: the active clients) with `model`, into the
    folder `output` (see the module's text). columns: values added to every row (forecaster, load_id, ...). prefix:
    the subfolder of `name` holding the daily flows (cashflow.read_manifest; "" in production). Returns
    the seconds of each step (added up over the groups), the wall time, the counts per status, the most memory used
    and the output's subfolder."""
    from concurrent.futures import ThreadPoolExecutor
    import dataiku
    started = time.time()
    folder, manifest = read_manifest(name, prefix)
    if manifest is None:
        raise ValueError(f"{name} is empty: run the staging first")
    clients = (clients if isinstance(clients, pl.DataFrame) else pl.from_pandas(clients[["client_no"]])).select(
        pl.col("client_no").cast(pl.Utf8)).unique()
    clients = clients.join(read_first_days(folder, manifest), on="client_no", how="left").with_columns(
        group=client_groups(clients["client_no"], manifest["groups"]))
    by_group = {g: part.drop("group") for (g,), part in clients.group_by(["group"])}
    version = f"forecast_{cutoff:%Y-%m-%d}_{model_id}"
    out = dataiku.Folder(output)
    for path in out.list_paths_in_partition():                  # an earlier try of the same night
        if path.startswith(f"/{version}/"):
            out.delete_path(path)
    workers = max(1, min(int(workers), len(by_group)))
    model.threads = max(1, (os.cpu_count() or 1) // workers)    # CatBoost's threads: the cores shared out
    import threading
    _JOB.update(model=model, cutoff=cutoff, days=input_days(model), clients=by_group, folder=folder, writing=threading.Lock(),
                manifest=manifest, output=out, version=version,
                columns={"model_id": model_id, **(columns or {})})
    summary = {k: 0.0 for k in STEPS}
    summary.update(daily_rows=0, groups=len(by_group), workers=workers)
    progress = Progress(f"forecasting {len(clients):,} clients: {len(by_group):,} groups, {workers} at a time",
                        len(clients))
    with ThreadPoolExecutor(workers) as pool:
        for result in pool.map(forecast_group, sorted(by_group)):
            for k, v in result.items():
                if k != "clients":
                    summary[k] = summary.get(k, 0) + v
            progress.add(result["clients"])
    progress.finish()
    with out.get_writer(CURRENT) as writer:
        writer.write(version.encode())
    summary.update(clients=len(clients), seconds=time.time() - started, output=version, peak_memory_mb=peak_memory_mb())
    return summary


def current_output(output=OUTPUT):
    """The output folder, its current subfolder and that subfolder's Parquet files (one per group)."""
    import dataiku
    folder = dataiku.Folder(output)
    paths = folder.list_paths_in_partition()
    if CURRENT not in paths:
        raise ValueError(f"No forecast in the folder {output} yet: run the nightly forecast first")
    with open(local_path(folder, CURRENT)) as f:
        version = f.read().strip()
    return folder, version, sorted(p for p in paths if p.startswith(f"/{version}/") and p.endswith(".parquet"))


def load_to_sql(dataset, output=OUTPUT):
    """Load the current forecast of the output folder into the SQL dataset `dataset` (replacing what was there), one
    group's file at a time. Returns the rows loaded and the subfolder."""
    import dataiku
    folder, version, paths = current_output(output)
    sql, writer, rows = dataiku.Dataset(dataset), None, 0
    progress = Progress(f"loading {version} into {dataset} (files)", len(paths))
    try:
        for path in paths:
            frame = read_parquet(folder, path).to_pandas()
            if writer is None:
                sql.write_schema_from_dataframe(frame)
                writer = sql.get_writer()
            writer.write_dataframe(frame)
            rows += len(frame)
            progress.add()
    finally:
        if writer is not None:
            writer.close()
    progress.finish()
    return rows, version


# ── SQL tables written by code ─────────────────────────────────────────────────────────────────────────────────────
# A table made with SQL (create_table_sql, ensure_table), then filled with the current forecast (load_to_table). On
# Oracle (the bank) it is written with Oracle's own driver, oracledb, in batches with bind variables (about 16,000
# rows a second; plain INSERT statements through Dataiku: under 1,000, Oracle parses each one), and no Dataiku dataset
# is needed (the bank's licence caps them). Any other database (PostgreSQL in the sandbox): through Dataiku's writer
# into a Dataiku dataset linked to the table.
SQL_TYPES = {"oracle": {"string": "VARCHAR2(4000)", "bigint": "NUMBER(19)", "double": "NUMBER", "date": "DATE"},
             "other": {"string": "VARCHAR(4000)", "bigint": "BIGINT", "double": "DOUBLE PRECISION", "date": "DATE"}}
BATCH_ROWS = 5000       # rows sent to Oracle at a time


def table_name(schema, table):
    """"SCHEMA.TABLE", or "TABLE" when the schema is empty (the connection's own)."""
    schema, table = str(schema or "").strip(), str(table or "").strip()
    return f"{schema}.{table}" if schema else table


def sql_dialect(connection):
    """"oracle" when the Dataiku connection's database answers SELECT ... FROM DUAL (Oracle), else "other"."""
    import dataiku
    try:
        dataiku.SQLExecutor2(connection=connection).query_to_df("SELECT 1 AS one FROM DUAL")
        return "oracle"
    except Exception:
        return "other"


def run_sql(connection, statements, dialect=None):
    """Runs the statements on the Dataiku connection, then COMMIT (Dataiku runs them before a query that returns rows:
    SELECT 1). dialect: sql_dialect(connection), asked when not given."""
    import dataiku
    dialect = dialect or sql_dialect(connection)
    dataiku.SQLExecutor2(connection=connection).query_to_df(
        "SELECT 1 AS one FROM DUAL" if dialect == "oracle" else "SELECT 1 AS one",
        pre_queries=list(statements), post_queries=["COMMIT"])


def output_columns(extra=None):
    """{column: kind} of the forecast's rows, in their order (kinds: string, bigint, double, date): those of
    output_rows (client_no, cutoff_date, w1_start ... w3_end, status, history_days, value_columns()), then those
    forecast_all adds to every row: model_id, then `extra` (its `columns`, e.g. {"forecaster": "string"})."""
    dates = ["cutoff_date"] + [f"w{h}_{edge}" for h in HORIZONS for edge in ("start", "end")]
    return {"client_no": "string", **{d: "date" for d in dates}, "status": "string", "history_days": "bigint",
            **{c: "double" for c in value_columns()}, "model_id": "string", **(extra or {})}


def create_table_sql(table, columns, dialect="oracle"):
    """The CREATE TABLE statement of `table` ("SCHEMA.TABLE") with `columns` ({name: kind}, see SQL_TYPES)."""
    types = SQL_TYPES[dialect]
    body = ",\n".join(f"    {name} {types[kind]}" for name, kind in columns.items())
    return f"CREATE TABLE {table} (\n{body}\n)"


def _table_columns(connection, table, dialect):
    """The columns of `table` (lower case), read from the database's catalog, or None when it is not there."""
    import dataiku
    schema, _, name = table.rpartition(".")
    quoted = lambda v: "'" + v.replace("'", "''") + "'"  # noqa: E731
    if dialect == "oracle":         # unquoted names are stored in upper case
        query = (f"SELECT column_name FROM all_tab_columns WHERE table_name = UPPER({quoted(name)}) AND owner = "
                 + (f"UPPER({quoted(schema)})" if schema else "USER"))
    else:                           # PostgreSQL: in lower case
        query = (f"SELECT column_name FROM information_schema.columns WHERE LOWER(table_name) = LOWER({quoted(name)}) "
                 + (f"AND LOWER(table_schema) = LOWER({quoted(schema)})" if schema
                    else "AND table_schema = current_schema()"))
    found = dataiku.SQLExecutor2(connection=connection).query_to_df(query)
    names = {str(n).lower() for n in found.iloc[:, 0]} if len(found.columns) else set()
    return names or None


def ensure_table(connection, table, columns):
    """Creates `table` ("SCHEMA.TABLE") on the Dataiku connection when it is not there; when it is, adds the columns
    of `columns` ({name: kind}) it lacks (a newer version of the code). Returns a sentence saying what was done."""
    dialect = sql_dialect(connection)
    have = _table_columns(connection, table, dialect)
    if have is None:
        run_sql(connection, [create_table_sql(table, columns, dialect)], dialect)
        return f"created the table {table} on {connection}"
    missing = [c for c in columns if c.lower() not in have]
    if not missing:
        return f"the table {table} on {connection} is there, with every column"
    add = "ALTER TABLE {} ADD ({} {})" if dialect == "oracle" else "ALTER TABLE {} ADD COLUMN {} {}"
    run_sql(connection, [add.format(table, c, SQL_TYPES[dialect][columns[c]]) for c in missing], dialect)
    return (f"added {len(missing)} column{'s' if len(missing) > 1 else ''} to the table {table} on {connection}: "
            f"{', '.join(missing)}")


def _oracle_dsn(params):
    """The address of an Oracle Dataiku connection for oracledb, from its params: its JDBC URL (useURL:
    jdbc:oracle:thin:@//host:port/service, @host:port/service, @host:port:SID or @(DESCRIPTION=...)), else its host,
    port and db, which Dataiku joins as host:port:db (db: the SID)."""
    if params.get("useURL"):
        dsn = str(params.get("url") or "").strip()
        dsn = dsn.split("@", 1)[1] if "@" in dsn else dsn
    else:
        dsn = f"{params.get('host')}:{params.get('port') or 1521}:{params.get('db')}"
    dsn = dsn[2:] if dsn.startswith("//") else dsn
    parts = dsn.split(":")
    if not dsn.startswith("(") and "/" not in dsn and len(parts) == 3:          # host:port:SID
        return (f"(DESCRIPTION=(ADDRESS=(PROTOCOL=TCP)(HOST={parts[0]})(PORT={parts[1]}))"
                f"(CONNECT_DATA=(SID={parts[2]})))")
    return dsn


def _oracle_login(connection):
    """(user, password, address) of an Oracle Dataiku connection, read from Dataiku: the connection's details must be
    readable by one of the user's groups."""
    import dataiku
    ask = (f"Ask a Dataiku admin to add your group to 'Details readable by' of the connection {connection} "
           "(Administration > Connections > the connection > Security)")
    try:
        info = dataiku.api_client().get_connection(connection).get_info()
    except Exception as e:
        raise RuntimeError(f"Cannot read the details of the connection {connection} ({e}). {ask}.") from None
    params = info.get("params") or {}
    try:
        login = info.get_basic_credential()
    except Exception:           # not given: the params may hold them
        login = {"user": params.get("user"), "password": params.get("password")}
    if not login.get("user") or not login.get("password"):
        raise RuntimeError(f"Dataiku did not give the user and password of the connection {connection}. {ask}; if it "
                           "uses per-user credentials, enter yours (Profile > Credentials).")
    return login["user"], login["password"], _oracle_dsn(params)


def _oracle_connect(connection):
    """An oracledb connection (not in autocommit) with the login of the Oracle Dataiku connection."""
    import oracledb
    user, password, dsn = _oracle_login(connection)
    return oracledb.connect(user=user, password=password, dsn=dsn)


def check_table_writer(connection, driver=True):
    """For an Oracle connection, checks load_to_table can write there: the package oracledb is in this Python and logs
    in (driver=False: not checked, for a notebook running on another Python than the recipes), and the connection's
    user and password can be read. Raises a RuntimeError saying what to ask the admin. Returns a sentence, or None
    for another database (Dataiku's writer: nothing to check)."""
    if sql_dialect(connection) != "oracle":
        return None
    if driver:
        try:
            import oracledb
        except ImportError:
            raise RuntimeError("The package oracledb (Oracle's Python driver) is not in this code env: ask a Dataiku "
                               "admin to add oracledb to it. The forecast is written into the Oracle table with it "
                               "(no Dataiku dataset).") from None
    user, password, dsn = _oracle_login(connection)
    if not driver:
        return f"the login of {connection} can be read (user {user})"
    try:
        oracledb.connect(user=user, password=password, dsn=dsn).close()
    except Exception as e:
        raise RuntimeError(f"oracledb could not log in to {dsn} as {user}, the login of {connection} ({e}). With "
                           "DPY-3010 or DPY-3015 (an older Oracle), oracledb needs its thick mode: ask the admin for "
                           "the Oracle Instant Client on the Dataiku server.") from None
    return f"oracledb logs in to {connection} (user {user})"


def _kind(dtype):
    """The kind (SQL_TYPES) of a Polars column type."""
    name = str(dtype).lower()
    if name.startswith(("int", "uint")) or name == "boolean":
        return "bigint"
    return "double" if name.startswith("float") else "date" if name.startswith("date") else "string"


def load_to_table(connection, table, dataset=None, output=OUTPUT):
    """Loads the current forecast of the folder `output` (one Parquet file per group: current_output) into `table`
    ("SCHEMA.TABLE") on the Dataiku connection, replacing its rows, one group's file at a time. Returns the rows
    loaded and the subfolder, like load_to_sql.
        Oracle      oracledb (check_table_writer): the table is made or given the columns it lacks first
                    (ensure_table, with the Parquet's columns), then its rows are deleted and the new ones inserted in
                    batches of BATCH_ROWS, all in one transaction: readers see the old rows until the one commit at
                    the end, and a failure leaves them.
        other       Dataiku's writer into `dataset`, a Dataiku dataset linked to the table (needed), which keeps the
                    table made by SQL (its schema is not rewritten)."""
    folder, version, paths = current_output(output)
    if sql_dialect(connection) != "oracle":
        return _load_with_dataset(dataset, table, connection, folder, version, paths)
    import oracledb
    binds = {"string": oracledb.DB_TYPE_VARCHAR, "bigint": oracledb.DB_TYPE_NUMBER,
             "double": oracledb.DB_TYPE_NUMBER, "date": oracledb.DB_TYPE_DATE}
    if not paths:
        print(ensure_table(connection, table, output_columns()))
    rows, progress = 0, Progress(f"loading {version} into {table} on {connection} (files, oracledb)", len(paths))
    db = _oracle_connect(connection)
    try:
        cursor = db.cursor()
        for n, path in enumerate(paths):
            frame = read_parquet(folder, path)
            frame = frame.with_columns(pl.col(pl.Float32, pl.Float64).fill_nan(None))     # NaN: empty
            kinds = {c: _kind(t) for c, t in frame.schema.items()}
            if n == 0:
                print(ensure_table(connection, table, kinds), flush=True)
                cursor.execute(f"DELETE FROM {table}")          # undone if anything below fails
            insert = (f"INSERT INTO {table} ({', '.join(kinds)}) "
                      f"VALUES ({', '.join(f':{i}' for i in range(1, len(kinds) + 1))})")
            cursor.setinputsizes(*[binds[k] for k in kinds.values()])
            data = frame.rows()
            for start in range(0, len(data), BATCH_ROWS):
                cursor.executemany(insert, data[start:start + BATCH_ROWS])
            rows += len(data)
            progress.add()
        if not paths:
            cursor.execute(f"DELETE FROM {table}")
        db.commit()
    finally:
        db.close()                  # without the commit: rolled back
    progress.finish()
    return rows, version


def _load_with_dataset(dataset, table, connection, folder, version, paths):
    """load_to_table on a database other than Oracle: Dataiku's writer into `dataset`, linked to the table (its
    schema is not written: the table keeps the types it was made with)."""
    import dataiku
    if not dataset:
        raise ValueError(f"The table {table} on {connection} is not on Oracle: it is written through a Dataiku dataset "
                         "linked to it, and none was given (dataset=...)")
    # the writer matches columns by their exact name, and the database keeps unquoted names in its own case
    # (PostgreSQL: total_w1_ev for TOTAL_w1_ev): each column is renamed to the dataset's own name, case aside
    names = [c["name"] for c in dataiku.Dataset(dataset).read_schema()]
    by_lower = {n.lower(): n for n in names}
    writer, rows = None, 0
    progress = Progress(f"loading {version} into {table} on {connection} (files, Dataiku's writer)", len(paths))
    try:
        writer = dataiku.Dataset(dataset).get_writer()
        for path in paths:
            frame = read_parquet(folder, path).to_pandas()
            unknown = [c for c in frame.columns if c.lower() not in by_lower]
            if unknown:
                raise ValueError(f"The table {table} (dataset {dataset}) has no column {', '.join(unknown)}: run the "
                                 "setup notebook again (it adds the columns a newer version of the code writes)")
            frame = frame.rename(columns={c: by_lower[c.lower()] for c in frame.columns})
            frame = frame.reindex(columns=names)                # the table's order (a column not written: empty)
            writer.write_dataframe(frame)
            rows += len(frame)
            progress.add()
    finally:
        if writer is not None:
            writer.close()
    progress.finish()
    return rows, version
'''

In [ ]:
LOADTEST_CODE = r'''
"""Load test (project library of the load test project: python/loadtest.py). Times the nightly forecast of the
forecast project on fake data of the same shape, with the same code: cashflow.py, models.py and forecast.py are copies
of the forecast project's, made by the same setup files. Nothing here reads or writes the forecast project.

One scenario, "Run the load test". Its step 1 sets the tests, run one after the other in one go (step1_tests):
    {"tests": {"clients": [100000, 400000, 800000], "months": 24, "postings_per_month": 60,
               "clients_per_group": [2000, 5000], "workers": [2, 4, 8]}}
tests is a grid (each number, or a list of numbers: every combination is a test, the clients first, the workers last)
or a list of tests, each with every number, [{"clients": ..., "months": ..., ..., "workers": ...}, ...]. The numbers:
    clients              how many fake clients
    months               months of daily flows per client up to the day they are made (at least MIN_MONTHS)
    postings_per_month   the average number of postings per client and month
    clients_per_group    clients per group (production: cashflow.CLIENTS_PER_GROUP)
    workers              groups of clients forecast at a time (production: the project variable inference_workers)
Zone Data prep (recipe make_fake_data, not timed): the fake daily flows of each size listed (the first 4 numbers) in
load_test_flows, one subfolder per size (size_path), each laid out like daily_flows (one file per group of clients and
month), made only when not there yet; nothing is deleted by code (remove unused sizes by hand).
Zone Load test (recipe run_load_test, then load_test_report): each test on its size's data: the placeholder model
(models.DailyCatBoost) fitted on 2 clients only (its forecasts mean nothing, its prediction time is real), the nightly
forecast of every fake client (forecast.forecast_all) into the folder load_test_forecast, then its final load, timed
(run_test), in its own process (in_own_process). The final load is set by FINAL_LOAD in the setup notebook (the project
variable final_load): "csv", one CSV file in the folder load_test_forecast (load_to_file: no table, no dataset), or
"sql", the table FINAL_SCHEMA.FINAL_TABLE on FINAL_CONNECTION (the project variables final_*: final_table), made by
the setup notebook with SQL and loaded by forecast.load_to_table: on Oracle with oracledb (no Dataiku dataset), on
another database through the dataset load_test_final linked to it."""
import itertools
import json
import multiprocessing
import os
import time
import traceback

import numpy as np
import pandas as pd

from cashflow import INPUT_DAYS, PERIOD_DAYS, Progress, client_groups, local_path, whole_number, write_daily
from warehouse import CATEGORY_OF_TYPE, COLUMNS, OTHER

SCENARIO = "Run the load test"
FLOWS = "load_test_flows"         # folder: the fake daily flows, laid out like daily_flows
OUTPUT = "load_test_forecast"     # folder: the forecast, like the folder forecast
SQL = "load_test_final"           # final load "sql", not on Oracle: the dataset linked to the table (final_table)
FINAL_FILE = "/final/load_test_final.csv"    # final load "csv": this file of the folder OUTPUT
MIN_MONTHS = -(-(INPUT_DAYS + PERIOD_DAYS * 3) // PERIOD_DAYS)    # a year of input and 3 windows: 16 months
SIZE = ("clients", "months", "postings_per_month", "clients_per_group")     # the fake data's size
TEST = SIZE + ("workers",)                                                  # the numbers of one test


def check(test, n=1):
    """One test of step 1 (TEST: whole numbers, months at least MIN_MONTHS), checked. n: its place in the list."""
    if not isinstance(test, dict):
        raise ValueError(f"test {n} must be an object {{\"clients\": ..., ...}}: set it in step 1 of the scenario "
                         f"{SCENARIO}")
    checked = {}
    for name in TEST:
        checked[name] = whole_number(name, test.get(name))
        if checked[name] is None:
            raise ValueError(f"test {n}: {name} must be a whole number (1 or more): set it in step 1 of the scenario "
                             f"{SCENARIO}")
    if checked["months"] < MIN_MONTHS:
        raise ValueError(f"test {n}: months is {checked['months']}: it must be at least {MIN_MONTHS} (a year of input, "
                         f"{INPUT_DAYS} days, and the 3 windows after it)")
    return checked


def grid_tests(grid):
    """A grid {"clients": [100000, 400000], "workers": [2, 4], "months": 24, ...} (each a number or a list): every
    combination, as a list of tests, in the order of TEST (the clients first, the workers last)."""
    lists = {name: value if isinstance(value, list) else [value] for name, value in grid.items()}
    return [dict(zip(TEST, numbers)) for numbers in itertools.product(*(lists.get(name, [None]) for name in TEST))]


def check_tests(tests):
    """The tests of step 1 (a grid, or a list of tests), as a list, each checked."""
    if isinstance(tests, dict):
        tests = grid_tests(tests)
    if not isinstance(tests, list) or not tests:
        raise ValueError(f"tests must be a grid {{\"clients\": [...], \"workers\": [...], ...}} or a list of tests "
                         f"[{{\"clients\": ..., ...}}, ...]: set it in step 1 of the scenario {SCENARIO}")
    return [check(test, n) for n, test in enumerate(tests, 1)]


def step1_variables():
    """The variables set in step 1 of the scenario, read from its settings, so they are the same whether the scenario
    runs, a recipe is built from the Flow or the setup notebook runs it."""
    import dataiku
    project = dataiku.api_client().get_default_project()
    ids = {s["name"]: s["id"] for s in project.list_scenarios()}
    steps = (project.get_scenario(ids[SCENARIO]).get_settings().get_raw()["params"].get("steps", [])
             if SCENARIO in ids else [])
    return steps[0]["params"].get("variables", {}) if steps and steps[0].get("type") == "define_vars" else {}


def step1_tests():
    """The tests set in step 1 of the scenario, as a list, each checked."""
    return check_tests(step1_variables().get("tests"))


# load_test_log: its columns and their Dataiku types (string, bigint, double). run_load_test writes them into the
# dataset load_test_log, and into the SQL table set by the project variables log_sql_* when there is one (log_to_sql).
LOG_COLUMNS = {
    "batch": "string", "test": "bigint", "run_at": "string", "status": "string", "error": "string",
    "workers": "bigint", "clients": "bigint", "months": "bigint", "postings_per_month": "bigint",
    "clients_per_group": "bigint", "groups": "bigint", "model": "string", "input_days": "bigint",
    "cutoff_date": "string", "daily_rows": "bigint", "data_mb": "double", "seconds_fitting_2_clients": "double",
    **{f"seconds_{k}": "double" for k in ("reading", "preparing", "model", "formatting", "writing")},  # forecast.STEPS
    "seconds_forecast": "double", "seconds_sql_load": "double", "clients_per_second": "double",
    "peak_memory_mb": "bigint", "daily_rows_read": "bigint", "clients_ok": "bigint",
    "clients_not_enough_history": "bigint", "clients_no_postings": "bigint", "cpus": "bigint", "ram_gb": "double",
    "final_load_to": "string",
    "cpu_limit": "double", "ram_limit_gb": "double", "ram_free_gb": "double", "disk_gb": "double",
    "disk_free_gb": "double"}       # the machine: machine(), read from Linux as each test starts

# The SQL copy of load_test_log (log_to_sql): a table the project's user may create and insert into, set by the
# setup notebook (LOG_SQL_CONNECTION, LOG_SQL_SCHEMA, LOG_SQL_TABLE) as the project variables log_sql_*. Made and
# written with SQL statements on the connection (forecast.ensure_table, forecast.run_sql: dataiku.SQLExecutor2), not
# through a Dataiku dataset: no dataset is created. The tables' helpers are forecast.py's (shared with production).


def sql_target(prefix):
    """(connection, "SCHEMA.TABLE") set by the project variables <prefix>_connection, _schema, _table, or None when
    the connection or the table is empty."""
    import dataiku
    from forecast import table_name
    variables = dataiku.get_custom_variables()
    connection, schema, table = (str(variables.get(f"{prefix}_{k}") or "").strip()
                                 for k in ("connection", "schema", "table"))
    return (connection, table_name(schema, table)) if connection and table else None


def log_table():
    """(connection, "SCHEMA.TABLE") of the SQL copy of load_test_log, from the project variables, or None (not set)."""
    return sql_target("log_sql")


def ensure_log_table(connection, table, columns=None):
    """Creates the table when it is not there; when it is, adds the columns it lacks (a newer version of the code).
    Returns what was done."""
    from forecast import ensure_table
    return ensure_table(connection, table, columns or LOG_COLUMNS)


def sql_value(value, kind):
    """A value as an SQL literal: NULL when missing, a quoted string (quotes doubled, at most 1,000 characters), or a
    number."""
    try:
        if value is None or pd.isna(value):
            return "NULL"
    except (TypeError, ValueError):
        pass
    if kind == "string":
        text = str(value)[:1000].replace("'", "''")
        return f"'{text}'" if text else "NULL"
    number = float(value)
    if not np.isfinite(number):
        return "NULL"
    return str(int(round(number))) if kind == "bigint" else repr(number)


def log_to_sql(rows, columns=None):
    """Adds the rows (this run's tests) to the SQL copy of load_test_log, made first if needed (ensure_log_table).
    Returns what was done, or None when no table is set (the project variables log_sql_*)."""
    from forecast import run_sql
    target = log_table()
    if target is None:
        return None
    columns = columns or LOG_COLUMNS
    connection, table = target
    done = ensure_log_table(connection, table, columns)
    names = ", ".join(columns)
    run_sql(connection, [f"INSERT INTO {table} ({names}) VALUES "
                         f"({', '.join(sql_value(row.get(c), kind) for c, kind in columns.items())})"
                         for row in rows])
    return f"{done}; {len(rows)} rows added"


GB = 1024 ** 3


def _read(path):
    """A small file's text, stripped, or None when it can't be read."""
    try:
        with open(path) as f:
            return f.read().strip()
    except OSError:
        return None


def _cgroup_dirs(v1_controller):
    """The cgroup folders that limit this process for one controller (cgroup v2, or v1's v1_controller), its own first
    then each parent up to the root (a limit set on a parent holds too). Read from /proc/self/cgroup."""
    dirs = []
    for line in (_read("/proc/self/cgroup") or "").splitlines():
        parts = line.split(":", 2)
        if len(parts) != 3:
            continue
        _, controllers, path = parts
        if controllers == "":                                               # cgroup v2: one tree
            root = "/sys/fs/cgroup"
        elif v1_controller in controllers.split(","):                       # cgroup v1: one tree per controller
            root = next((r for r in (f"/sys/fs/cgroup/{controllers}", f"/sys/fs/cgroup/{v1_controller}")
                         if os.path.isdir(r)), None)
            if root is None:
                continue
        else:
            continue
        here = os.path.join(root, path.lstrip("/"))
        if not os.path.isdir(here):          # in a container the path is the host's: the container's cgroup is the root
            here = root
        while True:
            dirs.append(here)
            if os.path.normpath(here) == os.path.normpath(root):
                break
            here = os.path.dirname(here)
    return dirs


def cpu_limit():
    """The CPU limit of this process's cgroup (container), in CPUs: its CPU time quota over the period (cgroup v2
    cpu.max, v1 cpu.cfs_quota_us / cpu.cfs_period_us), the lowest of its own and its parents'. None: no limit."""
    limits = []
    for d in _cgroup_dirs("cpu"):
        v2 = (_read(os.path.join(d, "cpu.max")) or "").split()
        if len(v2) == 2 and v2[0].isdigit() and v2[1].isdigit() and int(v2[1]):
            limits.append(int(v2[0]) / int(v2[1]))
        quota, period = _read(os.path.join(d, "cpu.cfs_quota_us")), _read(os.path.join(d, "cpu.cfs_period_us"))
        if quota and period and quota.isdigit() and period.isdigit() and int(period):    # -1 (no limit) is not digits
            limits.append(int(quota) / int(period))
    return round(min(limits), 2) if limits else None


def _meminfo():
    """/proc/meminfo in bytes: {"MemTotal": ..., "MemAvailable": ..., ...}."""
    info = {}
    for line in (_read("/proc/meminfo") or "").splitlines():
        name, _, value = line.partition(":")
        if value.split() and value.split()[0].isdigit():
            info[name] = int(value.split()[0]) * 1024
    return info


def memory_limit():
    """The memory limit of this process's cgroup (container) and the memory free under it, in bytes: (limit, free),
    the cgroup with the lowest limit (v2 memory.max, v1 memory.limit_in_bytes; a v1 'limit' above the server's RAM
    means none). Free = the limit - what the cgroup uses, not counting the file cache it can drop (inactive_file, as
    docker stats counts it). (None, None): no limit."""
    total = _meminfo().get("MemTotal")
    best = (None, None)
    for d in _cgroup_dirs("memory"):
        limit, used = _read(os.path.join(d, "memory.max")), _read(os.path.join(d, "memory.current"))
        stat_name = "inactive_file"
        if limit is None:
            limit, used, stat_name = (_read(os.path.join(d, "memory.limit_in_bytes")),
                                      _read(os.path.join(d, "memory.usage_in_bytes")), "total_inactive_file")
        if not (limit and limit.isdigit()) or (total and int(limit) >= total):
            continue
        stat = dict(line.split() for line in (_read(os.path.join(d, "memory.stat")) or "").splitlines()
                    if len(line.split()) == 2)
        cache = int(stat.get(stat_name, 0)) if str(stat.get(stat_name, 0)).isdigit() else 0
        free = int(limit) - (int(used) - cache) if used and used.isdigit() else None
        if best[0] is None or int(limit) < best[0]:
            best = (int(limit), free)
    return best


def machine(folder=OUTPUT):
    """The machine, read from Linux when a test starts (nothing is set by hand):
        cpus           CPUs this process may run on (os.sched_getaffinity, else os.cpu_count)
        cpu_limit      its container's CPU limit, in CPUs (cpu_limit), empty when there is none
        ram_gb         the server's RAM (/proc/meminfo MemTotal)
        ram_limit_gb   its container's memory limit (memory_limit), empty when there is none
        ram_free_gb    the RAM free for it: /proc/meminfo MemAvailable, or less when the container's limit leaves less
        disk_gb, disk_free_gb   the size and free space of the disk of the folder `folder` (shutil.disk_usage)
    Each is empty when it can't be read."""
    import shutil
    cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
    info = _meminfo()
    limit, limit_free = memory_limit()
    free = [v for v in (info.get("MemAvailable"), limit_free) if v is not None]
    disk = (None, None)
    try:
        import dataiku
        usage = shutil.disk_usage(local_path(dataiku.Folder(folder)))
        disk = (usage.total, usage.free)
    except Exception as e:
        print(f"machine: the disk of the folder {folder} could not be read: {e}", flush=True)
    gb = lambda v: round(v / GB, 1) if v is not None else None  # noqa: E731
    return {"cpus": cpus, "cpu_limit": cpu_limit(), "ram_gb": gb(info.get("MemTotal")), "ram_limit_gb": gb(limit),
            "ram_free_gb": gb(min(free)) if free else None, "disk_gb": gb(disk[0]), "disk_free_gb": gb(disk[1])}


def size_path(test):
    """The subfolder of load_test_flows holding the fake data of a test's size."""
    return (f"/{test['clients']}_clients_{test['months']}_months_{test['postings_per_month']}_postings_"
            f"{test['clients_per_group']}_per_group")


def data_info(test):
    """What fake data of this test's size is in load_test_flows (its data.json: size, cut-off day, rows, MB), or
    None."""
    import dataiku
    path = local_path(dataiku.Folder(FLOWS), f"{size_path(test)}/data.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def in_own_process(function, *args):
    """function(*args) in a forked process: its memory is its own (measured on its own, freed at its end), and a
    crash (out of memory, for example) stops it alone. Returns ("ok", its result) or ("error", the error, details)."""
    context = multiprocessing.get_context("fork")
    receive, send = context.Pipe(duplex=False)

    def target():
        try:
            send.send(("ok", function(*args)))
        except BaseException as e:
            send.send(("error", f"{type(e).__name__}: {e}", traceback.format_exc()))
        finally:
            send.close()

    process = context.Process(target=target)
    process.start()
    send.close()                        # only the child sends: when it ends without a word, recv stops
    try:
        result = receive.recv()
    except EOFError:
        result = None
    process.join()
    if result is None:
        code = process.exitcode
        why = " (killed: most likely out of memory)" if code in (-9, 137) else ""
        result = ("error", f"the process stopped with code {code}{why}", "")
    return result


def fake_clients(n):
    """The fake clients: client_no LT00000000, LT00000001, ..., all ACTIVE (the columns of the clients dataset)."""
    clients = pd.DataFrame({column: "LOAD TEST" for column in COLUMNS["clients"]}, index=range(n))
    clients["client_no"] = np.array([f"LT{i:08d}" for i in range(n)])
    clients["status"] = "ACTIVE"
    return clients


def fake_data(wanted, cutoff):
    """Zone Data prep: the fake daily flows of the size `wanted` (SIZE) in their subfolder of load_test_flows
    (size_path): kept when they are already there (whatever their cut-off day), else made up to `cutoff`. Returns what
    is there (data.json). Each client gets its own rate around the average (a few busy clients, many quiet ones), and 10% of
    them a shorter history, so some get NOT_ENOUGH_HISTORY, as in production."""
    import dataiku
    started = time.time()
    wanted, prefix = {k: wanted[k] for k in SIZE}, size_path(wanted)
    info = data_info(wanted)
    if info:
        print(f"PROGRESS fake data {prefix[1:]}: already there (up to {info['cutoff']}): kept", flush=True)
        return info
    folder = dataiku.Folder(FLOWS)
    clients, months, postings = wanted["clients"], wanted["months"], wanted["postings_per_month"]
    groups = max(1, -(-clients // wanted["clients_per_group"]))
    days = months * PERIOD_DAYS
    categories = np.array(sorted(set(CATEGORY_OF_TYPE.values()) | {OTHER}))
    numbers = fake_clients(clients)["client_no"].to_numpy()
    group_of = client_groups(numbers, groups).to_numpy()
    rng = np.random.default_rng(7)
    totals = {"postings": 0, "rows": 0}
    print(f"PROGRESS fake data {prefix[1:]}: {clients:,} clients ({groups:,} groups) x {months} months x about {postings} postings "
          f"a month = about {clients * months * postings:,} postings, up to {cutoff:%Y-%m-%d}", flush=True)

    def parts():
        """The fake daily flows, one group of clients at a time."""
        progress = Progress("fake data: writing the daily flows (groups of clients)", groups)
        for g in range(groups):
            who = numbers[group_of == g]
            n = len(who)
            rate = postings * rng.gamma(2.0, 0.5, n)                       # postings a month, average `postings`
            span = np.where(rng.random(n) < 0.1, rng.integers(30, days + 1, n), days)   # days of history
            client = np.repeat(np.arange(n), rng.poisson(rate * span / PERIOD_DAYS))
            amount = rng.lognormal(8, 1.2, n)[client] * rng.lognormal(0, 0.8, len(client))     # SAR
            money_in = rng.random(len(client)) < 0.55
            one = pd.DataFrame({
                "client": client,
                "back": (rng.random(len(client)) * span[client]).astype(int),     # days before the cut-off day
                "category": rng.integers(0, len(categories), len(client)),
                "money_in": np.where(money_in, amount, 0.0), "money_out": np.where(money_in, 0.0, amount)})
            daily = (one.groupby(["client", "back", "category"])
                     .agg(money_in=("money_in", "sum"), money_out=("money_out", "sum"), postings=("money_in", "size"))
                     .reset_index())
            totals["postings"] += len(one)
            totals["rows"] += len(daily)
            progress.add()
            yield pd.DataFrame({"client_no": who[daily["client"].to_numpy()],
                                "day": (cutoff - pd.to_timedelta(daily["back"].to_numpy(), unit="D")).values,
                                "category": categories[daily["category"].to_numpy()],
                                "money_in": daily["money_in"].round(2).to_numpy(),
                                "money_out": daily["money_out"].round(2).to_numpy(),
                                "postings": daily["postings"].to_numpy()})
        progress.finish()

    manifest = write_daily(parts(), groups, FLOWS, prefix=prefix)
    info = {**wanted, "groups": groups, "cutoff": f"{cutoff:%Y-%m-%d}", "postings": totals["postings"],
            "daily_rows": manifest["rows"], "files": sum(len(m) for m in manifest["files"].values()),
            "data_mb": round(sum(folder.get_path_details(p).get("size", 0)
                                 for p in folder.list_paths_in_partition() if p.startswith(prefix + "/")) / 1e6, 1),
            "seconds_generating": round(time.time() - started, 1)}
    with folder.get_writer(f"{prefix}/data.json") as writer:
        writer.write(json.dumps(info).encode())
    print(f"PROGRESS fake data: done in {info['seconds_generating']:.0f} s: {info['postings']:,} postings, "
          f"{info['daily_rows']:,} daily rows, {info['files']:,} files ({info['data_mb']} MB)", flush=True)
    return info


def prepare_data(tests, cutoff):
    """Zone Data prep: the fake data of every size the tests use (fake_data), made only when not there yet.
    Nothing is deleted: remove the sizes no longer needed by hand (the folder load_test_flows).
    Returns {size_path: data.json}."""
    sizes = {size_path(t): t for t in tests}
    return {prefix: fake_data(test, cutoff) for prefix, test in sizes.items()}


def final_load():
    """The final load of the tests, "csv" or "sql": the project variable final_load (set by the setup notebook)."""
    import dataiku
    return dataiku.get_custom_variables().get("final_load", "csv")


def final_table():
    """The final load "sql": (connection, "SCHEMA.TABLE", the Dataiku dataset linked to it or None, how it is
    written), from the project variables final_* (FINAL_CONNECTION, FINAL_SCHEMA, FINAL_TABLE of the setup notebook).
    Oracle: oracledb, no dataset; another database: Dataiku's writer into the dataset load_test_final (SQL)."""
    from forecast import sql_dialect
    target = sql_target("final")
    if target is None:
        raise ValueError("The final load is \"sql\" but FINAL_CONNECTION or FINAL_TABLE is empty: set them in the "
                         "setup notebook and run it again")
    connection, table = target
    oracle = sql_dialect(connection) == "oracle"
    return connection, table, None if oracle else SQL, "oracledb in bulk" if oracle else "Dataiku's writer"


def load_to_file(output=OUTPUT):
    """The final load "csv": the current forecast of the folder `output` (one Parquet file per group) into one CSV
    file of the same folder (FINAL_FILE), replacing the previous one, one group's file at a time. Polars writes the
    CSV (in Rust, on several threads): pandas' to_csv took most of the final load turning the numbers into text."""
    from cashflow import read_parquet
    from forecast import current_output
    folder, version, paths = current_output(output)
    with folder.get_writer(FINAL_FILE) as writer:
        for n, path in enumerate(paths):
            writer.write(csv_bytes(read_parquet(folder, path), header=n == 0))


def csv_bytes(frame, header=True):
    """A Polars DataFrame as CSV bytes (dates as 2026-11-10, empty for a missing value, like pandas wrote them)."""
    try:
        return frame.write_csv(include_header=header).encode()
    except TypeError:                         # Polars before 0.19.17 names it has_header
        return frame.write_csv(has_header=header).encode()


def run_test(test, info):
    """Zone Load test: one test, on the fake data of its size in load_test_flows (info: its data.json): fit option A
    on 2 clients, forecast every fake client as the nightly forecast does with `workers`, then load the forecast into
    the CSV file or the SQL table (final_load, final_table). Returns the values for load_test_log."""
    from cashflow import read_group, read_manifest
    from forecast import STEPS, check_table_writer, forecast_all, load_to_table
    from models import DailyCatBoost
    to_csv = final_load() == "csv"
    if not to_csv:                  # before the forecast: a table that can't be written fails the test at once
        connection, table, dataset, how = final_table()
        check_table_writer(connection)
    cutoff = pd.Timestamp(info["cutoff"])
    print("PROGRESS fitting the model on 2 clients only (no real training: only the forecast is timed)", flush=True)
    ticked = time.time()
    prefix = size_path(test)
    folder, manifest = read_manifest(FLOWS, prefix)
    daily = read_group(folder, manifest, next(iter(manifest["files"]))).to_pandas()          # one group of clients
    two = daily.groupby("client_no")["day"].min().nsmallest(2).index                       # its 2 oldest clients
    model = DailyCatBoost().fit(daily[daily["client_no"].isin(two)], None, cutoff)
    log = {"model": model.name, "input_days": model.params["input_days"],
           "seconds_fitting_2_clients": round(time.time() - ticked, 1)}
    del daily

    done = forecast_all(model, "load_test", fake_clients(info["clients"]), cutoff, test["workers"], FLOWS, OUTPUT,
                        {"forecaster": model.name, "load_id": "load_test"}, prefix)
    ticked = time.time()
    if to_csv:
        load_to_file(OUTPUT)
        log["final_load_to"] = f"a CSV file ({OUTPUT}{FINAL_FILE})"
    else:
        load_to_table(connection, table, dataset, OUTPUT)
        log["final_load_to"] = f"the table {table} on {connection} ({how})"
    seconds_sql = time.time() - ticked
    log.update({f"seconds_{k}": round(done[k], 1) for k in STEPS}, groups=done["groups"],
               daily_rows_read=done["daily_rows"], seconds_forecast=round(done["seconds"], 1),
               seconds_sql_load=round(seconds_sql, 1), clients_ok=done.get("OK", 0),
               clients_not_enough_history=done.get("NOT_ENOUGH_HISTORY", 0), clients_no_postings=done.get("NO_POSTINGS", 0),
               clients_per_second=round(info["clients"] / max(done["seconds"], 0.1), 1),
               peak_memory_mb=done["peak_memory_mb"])
    return log
'''

In [ ]:
RECIPES = {}

In [ ]:
RECIPES["make_fake_data"] = r'''
# Load test, zone Data prep (not timed): the fake daily flows the tests read, in load_test_flows, one subfolder per
# size of fake data (clients, months, postings_per_month, clients_per_group) used by the tests in step 1 of the
# scenario Run the load test, each laid out like the forecast project's daily_flows (one Parquet file per group of
# clients and month). A size already there is kept (whatever day it was made); nothing is deleted (remove unused sizes by hand).
import pandas as pd

from loadtest import prepare_data, step1_tests

cutoff = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)          # yesterday, for new data
for prefix, info in prepare_data(step1_tests(), cutoff).items():
    print(f"PROGRESS fake data ready, {prefix[1:]}: up to {info['cutoff']}, {info['daily_rows']:,} daily rows, "
          f"{info['data_mb']} MB", flush=True)
'''

In [ ]:
RECIPES["run_load_test"] = r'''
# Load test, zone Load test: every test set in step 1 of the scenario Run the load test, one after the other, each
# on the fake data of its size in load_test_flows (made by the step before, not timed), each in its own process
# (forked from this one: its memory is measured on its own, and a crash such as running out of memory is logged as
# FAILED and the next test still runs). Adds one row per test to load_test_log, all with the same batch (this run),
# written together at the end.
# A test (loadtest.run_test): the placeholder model fitted on 2 clients, then the nightly forecast of every fake
# client with the forecast project's own code (forecast.forecast_all: `workers` groups of clients at a time, each
# read, prepared, forecast and written on its own), then the final load (a CSV file, or the table FINAL_SCHEMA.
# FINAL_TABLE on FINAL_CONNECTION, as set by FINAL_LOAD and FINAL_* in the setup notebook: loadtest.run_test; on Oracle
# with oracledb, else through the dataset load_test_final, an output of this recipe then). Its seconds:
#   reading, preparing, model, formatting, writing   added up over the groups (so over the workers too)
#   forecast                                         from start to end of the forecast (the wall clock)
#   sql_load                                         the final load (the CSV file, or the table)
import pandas as pd

from cashflow import append_row
from forecast import STEPS
from loadtest import LOG_COLUMNS, data_info, in_own_process, log_to_sql, machine, run_test, size_path, step1_tests

LOG = LOG_COLUMNS        # load_test_log's columns (loadtest.py), also those of its SQL copy (log_to_sql)
assert all(f"seconds_{k}" in LOG for k in STEPS), "loadtest.LOG_COLUMNS lacks a step of forecast.STEPS"
tests = step1_tests()
batch = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
rows, failed = [], []
for n, test in enumerate(tests, 1):
    row = {"batch": batch, "test": n, "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"), **test, **machine()}
    print(f"PROGRESS test {n}/{len(tests)}: machine at the start: " + ", ".join(
        f"{k} {row[k]}" for k in ("cpus", "cpu_limit", "ram_gb", "ram_limit_gb", "ram_free_gb", "disk_gb",
                                  "disk_free_gb")), flush=True)
    print(f"PROGRESS test {n}/{len(tests)}: {test['workers']} workers, {test['clients']:,} clients, {test['months']} "
          f"months, {test['postings_per_month']} postings a month, {test['clients_per_group']:,} per group", flush=True)
    info = data_info(test)
    if info is None:
        result = ("error", f"no fake data {size_path(test)[1:]} in load_test_flows: run the scenario Run the load "
                           "test (its step 2 makes it), not the recipe alone", "")
    else:
        row.update({k: info.get(k) for k in ("daily_rows", "data_mb")}, cutoff_date=info["cutoff"])
        result = in_own_process(run_test, test, info)
    if result[0] == "ok":
        row.update(result[1], status="SUCCESS")
        print(f"PROGRESS test {n}/{len(tests)}: forecast {row['seconds_forecast']:.0f} s + final load "
              f"{row['seconds_sql_load']:.0f} s, {row['clients_per_second']:,.0f} clients a second, "
              f"{row['peak_memory_mb']:,} MB at most", flush=True)
    else:
        row.update(status="FAILED", error=result[1][:1000])
        failed.append(f"test {n}: {result[1][:300]}")
        print(f"PROGRESS test {n}/{len(tests)}: FAILED: {result[1]}\n{result[2]}", flush=True)
    rows.append(row)
append_row("load_test_log", LOG, rows)
try:                                # the SQL copy of the log, when the setup notebook set one (LOG_SQL_*)
    done = log_to_sql(rows, LOG)
    print(f"PROGRESS the SQL log: {done or 'none set (LOG_SQL_CONNECTION in the setup notebook)'}", flush=True)
except Exception as e:              # the rows are in load_test_log all the same; the job fails below to show it
    failed.append(f"the SQL log could not be written ({type(e).__name__}: {str(e)[:500]})")
    print(f"PROGRESS the SQL log: FAILED: {e}", flush=True)
if failed:
    raise RuntimeError(f"{len(failed)} problem(s) (the report still runs): " + "; ".join(failed))
'''

In [ ]:
RECIPES["load_test_report"] = r'''
# Load test, the report of the last run of the scenario Run the load test (its tests: the rows of load_test_log with
# the same batch). The number of clients is not a setting (it is how many clients the bank has), so each client count
# is shown on its own, and only the settings (workers, clients per group) are compared, at the same client count:
#   1. the run: what a test times, the model, the fake data, the machine (CPUs, RAM)
#   2. one section per client count: a table (one line per setting: the forecast's 5 parts in seconds, the forecast,
#      the final load, the total, memory, compared with the best setting at this client count) and a chart of the
#      same times. The parts are measured added up over the workers (they run at the same time), so each part is
#      shown as its share of that work times the forecast's own time: the 5 add up to the forecast.
#   3. where the forecast's time goes, per setting, at the largest client count
#   4. each setting's time per 100,000 clients, at each client count
#   5. the earlier tests, as a table
# In the folder load_test_report: report_<batch>.html (one per run, kept) and latest_report.html, one page each; and
# the dashboard "Load test" (Dashboards > Load test), made again each time (do not edit it by hand: copy it first).
# A test's time: the forecast from start to end (the wall clock) plus the final load (a CSV file or the SQL table
# load_test_final, as set by FINAL_LOAD in the setup notebook); not making the fake data nor
# fitting the model on 2 clients (production does neither at night).
import base64
import html
import io

import cashflow  # noqa: F401  (quieter job log: hides Dataiku's writer chatter, as in every recipe)
import dataiku
import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from dataiku import insights  # noqa: E402

DASHBOARD = "Load test"
EARLIER = 20                    # the earlier tests shown
PER = 100_000                   # section 4: the time per this many clients
STEPS = {"seconds_reading": "reading", "seconds_preparing": "preparing", "seconds_model": "model",
         "seconds_formatting": "formatting", "seconds_writing": "writing"}
STEP_COLORS = ["#2a6f97", "#61a5c2", "#e07a5f", "#f2cc8f", "#81b29a"]
FINAL_COLOR = "#7d6b9e"         # the final load, after the forecast's 5 parts (STEP_COLORS)
NUMBERS = ["test", "workers", "clients", "months", "postings_per_month", "clients_per_group", "seconds_forecast",
           "seconds_sql_load", "peak_memory_mb", "cpus", "ram_gb", "cpu_limit", "ram_limit_gb", "ram_free_gb", "disk_gb",
           "disk_free_gb", *STEPS]
CELL = "padding:5px 9px;border-bottom:1px solid #ddd;text-align:left"
BEST = "background:#e8f5e9;font-weight:bold"
NOTE = "margin:2px 0 8px;color:#555;font-size:13px"

log = dataiku.Dataset("load_test_log").get_dataframe()
if log.empty:
    raise ValueError("load_test_log is empty: run the scenario Run the load test")
for column in NUMBERS:
    log[column] = pd.to_numeric(log[column], errors="coerce") if column in log else float("nan")
log["batch"] = log["batch"].fillna("").astype(str) if "batch" in log else ""
log["batch"] = log["batch"].where(log["batch"] != "", log["run_at"])      # tests from before batches: one run each
log["test"] = log["test"].fillna(1)
log["seconds_total"] = log["seconds_forecast"] + log["seconds_sql_load"]
log = log.sort_values(["run_at", "test"]).reset_index(drop=True)
batch_id = log.iloc[-1]["batch"]
run = log[log["batch"] == batch_id].sort_values("test")             # this run's tests, in their order
earlier = log[log["batch"] != batch_id].tail(EARLIER).iloc[::-1]     # the newest first
ok = run[run["status"] == "SUCCESS"]
data_varies = len(run[["months", "postings_per_month"]].drop_duplicates()) > 1
folder = dataiku.Folder("load_test_report")
dashboard_error = []        # why the dashboard could not be made (the HTML report is made all the same)


def to_dashboard(call, *args, **kwargs):
    """A call that makes the dashboard (an insight): when Dataiku refuses it (a licence limit, a right), the report
    goes on without the dashboard."""
    if dashboard_error:
        return
    try:
        call(*args, **kwargs)
    except Exception as e:
        dashboard_error.append(f"{type(e).__name__}: {e}")
        print(f"PROGRESS the dashboard is not made ({dashboard_error[0][:300]}): the HTML report still is", flush=True)


def duration(seconds):
    """3.2 s, 45 s, 2.5 min, 1.2 h."""
    if pd.isna(seconds):
        return ""
    return (f"{seconds:,.1f} s" if seconds < 10 else f"{seconds:,.0f} s" if seconds < 90
            else f"{seconds / 60:,.1f} min" if seconds < 5400 else f"{seconds / 3600:,.1f} h")


def gb(mb):
    return "" if pd.isna(mb) else f"{mb / 1024:,.1f} GB"


def setting(t, full=data_varies):
    """A test's setting: 4 workers · 5,000 per group (and its fake data's months and postings when they vary)."""
    workers = int(t["workers"])
    text = f"{workers} worker{'s' if workers != 1 else ''} · {int(t['clients_per_group']):,} per group"
    if full and pd.notna(t["months"]):
        text += f" · {int(t['months'])} months · {int(t['postings_per_month'])} postings/month"
    return text


def clock_parts(tests):
    """The forecast's 5 parts in seconds of the clock, one row per test: each part's share of the work (added up over
    the workers) times the forecast's own time, so they add up to the forecast."""
    work = tests[list(STEPS)].astype(float)
    return work.div(work.sum(axis=1).where(lambda total: total > 0), axis=0).mul(tests["seconds_forecast"], axis=0)


def table(headers, rows, styles=None):
    head = "".join(f"<th style='{CELL};background:#f3f5f7'>{h}</th>" for h in headers)
    body = "".join(f"<tr style='{(styles or {}).get(i, '')}'>" + "".join(f"<td style='{CELL}'>{v}</td>" for v in row)
                   + "</tr>" for i, row in enumerate(rows))
    return f"<table style='border-collapse:collapse;font-size:13px;margin:6px 0 14px'>{head}{body}</table>"


def failed(t):
    return f"<span style='color:#b00020'>FAILED: {html.escape(str(t['error'])[:200])}</span>"


def figure(fig, name):
    """The chart saved as an insight (for the dashboard) and a PNG in the folder; returns the PNG, base64."""
    fig.tight_layout()
    to_dashboard(insights.save_figure, f"load_test_{name}", fig, label=f"Load test: {name}")
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110)
    plt.close(fig)
    with folder.get_writer(f"{name}.png") as writer:
        writer.write(buffer.getvalue())
    return base64.b64encode(buffer.getvalue()).decode()


def image(png):
    return f"<p><img style='max-width:100%' src='data:image/png;base64,{png}'></p>"


def bar_chart(title, shown, xlabel):
    """A horizontal bar per test of `shown` (its setting), the first at the top. Returns the figure and its axes."""
    fig, ax = plt.subplots(figsize=(11, 1.6 + 0.45 * max(len(shown), 1)))
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(title, loc="left", fontsize=11, pad=24)
    ax.set_xlabel(xlabel)
    return fig, ax, [setting(t) for _, t in shown.iloc[::-1].iterrows()], shown.iloc[::-1]


# 1. the run
first = run.iloc[0]
model = html.escape(str(run.iloc[-1].get("model") or "daily_catboost"))


def gb(v):
    return f"{v / 1024:,.1f} TB" if v >= 1024 else f"{v:,.0f} GB" if v >= 10 else f"{v:,.1f} GB"


def recorded(column):
    """The column's value at the run's first test (the machine as the run started), None when not recorded (tests from
    before it was measured: then the report says so)."""
    return first[column] if pd.notna(first[column]) else None


cpus, cpu_cap = recorded("cpus"), recorded("cpu_limit")
ram, ram_cap, ram_free = recorded("ram_gb"), recorded("ram_limit_gb"), recorded("ram_free_gb")
disk, disk_free = recorded("disk_gb"), recorded("disk_free_gb")
measured = run[["ram_free_gb", "disk_gb"]].notna().any().any()     # this run's code measured the limits too
cpu_text = ("not recorded" if cpus is None else f"{int(cpus)} CPUs it may run on" + (
    f", but the job may use only {cpu_cap:g} CPUs' worth of time (its limit)" if cpu_cap is not None and cpu_cap < cpus
    else ", no CPU limit on the job" if measured else ""))
ram_text = ("not recorded" if ram is None else f"{gb(ram)} in the server" + (
    f", the job is limited to {gb(ram_cap)}" if ram_cap is not None
    else ", no memory limit on the job" if measured else "")
    + (f"; {gb(ram_free)} free when the run started" if ram_free is not None else ""))
disk_text = (f"{gb(disk_free)} free of {gb(disk)}, on the disk of the folder load_test_forecast"
             if disk is not None and disk_free is not None else "not recorded")
machine = (f"<ul style='margin:2px 0 0'><li>CPUs: {cpu_text}</li><li>RAM: {ram_text}</li><li>Disk: {disk_text}</li>"
           "</ul><span style='color:#555'>Read from Linux as each test starts (/proc, the job's cgroup: the limits set "
           "on its container or service, the folder's disk), the values of the first test. The server's other jobs share them.</span>")
final_to = sorted({str(v) for v in run.get("final_load_to", pd.Series(dtype=str)).dropna() if str(v)})
final_to = html.escape(" / ".join(final_to) or "the SQL table")
data = ("as in each setting" if data_varies else
        f"{int(first['months'])} months of daily flows per client, about {int(first['postings_per_month'])} postings "
        "a month per client")
header = (
    f"<h2 style='margin:0 0 4px'>Load test, {html.escape(str(batch_id))}: {len(run)} test{'s' if len(run) > 1 else ''}"
    "</h2>"
    "<p style='margin:0 0 6px;color:#555'>Each test: the nightly forecast of every fake client, then its final load "
    f"into {final_to}. <b>Total = forecast + final load</b> (making the fake data is not timed). The tests ran one "
    "after the other on the same machine.</p>"
    "<p style='margin:0 0 6px'><b>How to read it:</b> the number of clients is not a setting, it is how many clients "
    "the bank has. So each client count has its own table, and only the settings (workers, clients per group) are "
    "compared there.</p>"
    f"<ul style='margin:0 0 10px;font-size:13px'><li>Machine: {machine}</li><li>Fake data: {data}</li>"
    f"<li>Model: {model}, fitted on 2 clients only (its forecasts mean nothing, its time is real)</li></ul>")
if len(run) > len(ok):
    header += f"<p style='color:#b00020'>{len(run) - len(ok)} of {len(run)} tests failed: see the tables.</p>"
if not len(ok):
    header += "<p style='color:#b00020'>No test of this run succeeded.</p>"

# 2. one section per client count
sections = []          # (client count, its table's HTML, its chart's PNG or None, its number of tests)
for clients, tests in run.groupby("clients", sort=True):
    good = tests[tests["status"] == "SUCCESS"]
    best = good["seconds_total"].min() if len(good) else None
    parts = clock_parts(tests)
    lines, styles = [], {}
    for i, (index, t) in enumerate(tests.iterrows()):
        if t["status"] != "SUCCESS":
            lines.append([html.escape(setting(t)), failed(t)] + [""] * (len(STEPS) + 5))
            continue
        is_best = t["seconds_total"] == best
        if is_best:
            styles[i] = BEST
        lines.append([html.escape(setting(t)), *[duration(parts.loc[index, c]) for c in STEPS],
                      f"<b>{duration(t['seconds_forecast'])}</b>", duration(t["seconds_sql_load"]),
                      f"<b>{duration(t['seconds_total'])}</b>", gb(t["peak_memory_mb"]),
                      "the best here" if is_best else f"{t['seconds_total'] / best - 1:.0%} longer"])
    section = (f"<h3 style='margin:22px 0 2px'>{int(clients):,} clients</h3>"
               f"<p style='{NOTE}'>Each setting with {int(clients):,} fake clients. Compared only with the other "
               "settings at this client count. The forecast = its 5 parts: reading the daily flows, preparing the "
               "model's input rows, the model's prediction, formatting one row per client, writing the Parquet files "
               "(each part: its share of the workers' work × the forecast's time, so they add up to the forecast). "
               "Then the final load.</p>"
               + table(["Setting", *[s.capitalize() for s in STEPS.values()], "Forecast", "Final load", "Total",
                        "Memory (most used)", "Compared with the best here"], lines, styles))
    png = None
    if len(good):
        fig, ax, names, shown = bar_chart(f"{int(clients):,} clients: the total time per setting, the forecast's 5 "
                                          "parts then the final load (shorter is better)", good, "seconds")
        left = pd.Series(0.0, index=shown.index)
        for (column, step), color in zip(STEPS.items(), STEP_COLORS):
            ax.barh(names, parts.loc[shown.index, column].fillna(0), left=left, color=color, label=step)
            left += parts.loc[shown.index, column].fillna(0)
        ax.barh(names, shown["seconds_sql_load"], left=shown["seconds_forecast"], color=FINAL_COLOR,
                label="final load")
        for y, total in enumerate(shown["seconds_total"]):
            ax.text(total, y, f"  {duration(total)}", va="center", fontsize=9)
        ax.legend(ncol=6, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8, frameon=False)
        png = figure(fig, f"{int(clients)}_clients")
    sections.append((int(clients), section, png, len(tests)))

# 3. where the forecast's time goes, at the largest client count
steps_png, steps_html, steps_rows = None, "", 0
if len(ok):
    largest = ok["clients"].max()
    shown_tests = ok[ok["clients"] == largest]
    fig, ax, names, shown = bar_chart(f"Where the forecast's time goes, per setting, at {int(largest):,} clients "
                                      "(share of the workers' time)", shown_tests, "% of the forecast's work")
    shares = shown[list(STEPS)].div(shown[list(STEPS)].sum(axis=1), axis=0) * 100
    left = pd.Series(0.0, index=shares.index)
    for (column, step), color in zip(STEPS.items(), STEP_COLORS):
        ax.barh(names, shares[column], left=left, color=color, label=step)
        for y, (value, start) in enumerate(zip(shares[column], left)):
            if value >= 6:
                ax.text(start + value / 2, y, f"{value:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if color in ("#2a6f97", "#e07a5f") else "black")
        left += shares[column]
    ax.set_xlim(0, 100)
    ax.legend(ncol=5, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8, frameon=False)
    steps_png, steps_rows = figure(fig, "steps"), len(shown_tests)
    steps_html = ("<h3 style='margin:22px 0 2px'>Where the forecast's time goes</h3>"
                  f"<p style='{NOTE}'>reading: the daily flows; preparing: the model's input rows; model: its "
                  "prediction; formatting: one row per client; writing: the forecast's files. The final load is not in "
                  "it.</p>")

# 4. each setting's time per 100,000 clients, at each client count
counts = sorted(int(c) for c in ok["clients"].unique())
rate_lines = []
for name, tests in ok.assign(setting=ok.apply(setting, axis=1)).groupby("setting", sort=False):
    per_count = tests.groupby("clients")["seconds_total"].mean()
    lost = sorted(int(t["clients"]) for _, t in run.iterrows() if t["status"] != "SUCCESS" and setting(t) == name)
    rate_lines.append([html.escape(name) + "".join(f"<br><span style='color:#b00020'>failed at {c:,} clients</span>"
                                                   for c in lost),
                       *[duration(per_count[c] / c * PER) if c in per_count.index else "" for c in counts],
                       gb(tests["peak_memory_mb"].max())])
rate_html = (
    f"<h3 style='margin:22px 0 2px'>Time per {PER:,} clients</h3>"
    f"<p style='{NOTE}'>A test's total time (forecast + final load) ÷ its clients × {PER:,}, for each setting at each "
    "client count. When the values in a row are close, the time grows in step with the clients.</p>"
    + (table(["Setting", *[f"At {c:,} clients" for c in counts], "Memory (most used)"], rate_lines)
       if rate_lines else "<p>No successful test.</p>"))


# 5. the earlier tests
def earlier_row(t):
    done = t["status"] == "SUCCESS"
    return [html.escape(str(t["batch"])), int(t["test"]),
            f"{int(t['clients']):,}" if pd.notna(t["clients"]) else "",
            html.escape(setting(t, full=True)) if pd.notna(t["workers"]) else "",
            duration(t["seconds_forecast"]), duration(t["seconds_sql_load"]),
            duration(t["seconds_total"]) if done else "", gb(t["peak_memory_mb"]), html.escape(str(t["status"]))]


earlier_html = (f"<h3 style='margin:22px 0 4px'>Earlier tests (the last {len(earlier)}, the newest first)</h3>"
                + (table(["Run", "Test", "Clients", "Setting", "Forecast", "Final load", "Total", "Memory", "Status"],
                         [earlier_row(t) for _, t in earlier.iterrows()]) if len(earlier) else "<p>None yet.</p>"))

# the page
page = ("<!doctype html><html><head><meta charset='utf-8'><title>Load test report</title></head>"
        "<body style='font-family:sans-serif;max-width:1200px;margin:20px auto;padding:0 16px'>"
        + header
        + "".join(section + (image(png) if png else "") for _, section, png, _ in sections)
        + (steps_html + image(steps_png) if steps_png else "")
        + rate_html + earlier_html
        + "<p style='color:#777;font-size:12px'>Every test: the dataset load_test_log.</p></body></html>")
name = f"report_{str(batch_id).replace(':', '').replace(' ', '_')}.html"
for path in (name, "latest_report.html"):
    with folder.get_writer(path) as writer:
        writer.write(page.encode())

# the dashboard: the same parts, one tile each, in the same order
tiles_wanted = []                   # (insight ID, height)


def html_insight(insight_id, content, title, height):
    to_dashboard(insights.save_data, insight_id, f"<div style='font-family:sans-serif'>{content}</div>".encode(),
                 "text/html", label=f"Load test: {title}")
    tiles_wanted.append((insight_id, height))


html_insight("load_test_summary", header, "this run", 10)
for clients, section, png, n in sections:
    html_insight(f"load_test_{clients}_table", section, f"{clients:,} clients", 6 + 2 * n)
    if png:
        tiles_wanted.append((f"load_test_{clients}_clients", 5 + 2 * n))
if steps_png:
    html_insight("load_test_steps_text", steps_html, "where the time goes", 4)
    tiles_wanted.append(("load_test_steps", 5 + 2 * steps_rows))
html_insight("load_test_per_100k", rate_html, f"time per {PER:,} clients", 6 + 2 * len(rate_lines))
html_insight("load_test_earlier", earlier_html, "earlier tests", 4 + 2 * len(earlier))

def tile(insight_id, top, height):
    return {"tileType": "INSIGHT", "insightId": insight_id, "insightType": "static_file",
            "tileParams": {"numDisplayedComments": 5}, "displayMode": "INSIGHT", "clickAction": "DO_NOTHING",
            "box": {"left": 0, "top": top, "width": 36, "height": height},
            "titleOptions": {"showTitle": "NO"}, "resizeImageMode": "FIT_SIZE", "autoLoad": True}


def make_dashboard():
    project = dataiku.api_client().get_default_project()
    existing = [d for d in project.list_dashboards(as_type="listitems") if d.name == DASHBOARD]
    dashboard = existing[0].to_dashboard() if existing else project.create_dashboard(DASHBOARD)
    tiles, top = [], 0
    for insight_id, height in tiles_wanted:
        tiles.append(tile(insight_id, top, height))
        top += height
    settings = dashboard.get_settings()
    settings.get_raw()["pages"] = [{"id": None, "title": "Last run", "show": True, "grid": {"tiles": tiles}}]
    settings.save()


to_dashboard(make_dashboard)
print(f"PROGRESS load test report: done: {name} and latest_report.html in the folder load_test_report"
      + (f", not the dashboard ({dashboard_error[0][:200]})" if dashboard_error else f", and the dashboard {DASHBOARD!r}"),
      flush=True)
'''

In [ ]:
import json
import sys
import time
import types

import pandas as pd

if FINAL_LOAD not in ("csv", "sql"):
    raise RuntimeError(f"FINAL_LOAD is {FINAL_LOAD!r}: set it to \"csv\" or \"sql\" in the settings cell above.")
placeholders = [name for name, value in (("CODE_ENV", CODE_ENV), ("FINAL_CONNECTION", FINAL_CONNECTION),
                                         ("FINAL_TABLE", FINAL_TABLE))
                if (not str(value).strip() or str(value).startswith("YOUR_"))
                and (name == "CODE_ENV" or FINAL_LOAD == "sql")]       # FINAL_*: only for the final load "sql"
if placeholders:
    raise RuntimeError(f"Fill in {' and '.join(placeholders)} in the settings cell above, then run again.")


def load_module(name, code):
    """A library file as a module of this notebook: the code it writes to the project library."""
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(code, module.__dict__)
    return module


for name, code in (("cashflow", LIBRARY_CODE), ("warehouse", WAREHOUSE_CODE), ("forecast", FORECAST_CODE),
                   ("loadtest", LOADTEST_CODE)):
    load_module(name, code)             # to check the tests and make the SQL tables below (this notebook has no polars)
CHECKED = sys.modules["loadtest"].check_tests(TESTS)          # checked before anything is built (as a list)

# the final load "sql": the table FINAL_SCHEMA.FINAL_TABLE, made here with SQL (forecast.ensure_table) with the
# columns of the forecast's rows. Oracle: written by the recipe with oracledb (no dataset); else: through a dataset
forecast = sys.modules["forecast"]
final_sql = forecast.table_name(FINAL_SCHEMA, FINAL_TABLE) if FINAL_LOAD == "sql" else None
final_oracle = FINAL_LOAD == "sql" and forecast.sql_dialect(FINAL_CONNECTION.strip()) == "oracle"
check_code_env(CODE_ENV, ("polars", "catboost") + (("oracledb",) if final_oracle else ()))
if FINAL_LOAD == "sql":
    final_columns = forecast.output_columns({"forecaster": "string", "load_id": "string"})
    try:
        print(f"Final load: {forecast.ensure_table(FINAL_CONNECTION.strip(), final_sql, final_columns)}")
    except Exception as e:
        print("For a DBA, the table to make:\n" + forecast.create_table_sql(final_sql, final_columns,
                                                                           "oracle" if final_oracle else "other"))
        raise RuntimeError(f"The table {final_sql} on {FINAL_CONNECTION} could not be made or read ({e}). Check "
                           "FINAL_CONNECTION (a SQL connection of this Dataiku) and FINAL_SCHEMA, and that the "
                           "connection's user may create the table; or ask a DBA to run the CREATE TABLE above; or set "
                           "FINAL_LOAD = \"csv\".") from None
    if final_oracle:            # the login; oracledb itself is checked in the code env above (not this notebook's)
        print("Final load: Oracle, written with oracledb; "
              + forecast.check_table_writer(FINAL_CONNECTION.strip(), driver=False))

try:                                                # needs the project's admin right
    metadata = project.get_metadata()
    metadata["label"] = PROJECT_NAME
    project.set_metadata(metadata)
except Exception as e:
    if not any(w in str(e).lower() for w in ("unauthorized", "forbidden", "permission", "not allowed")):
        raise
    print(f"Warning: could not name the project {PROJECT_NAME!r} (no admin right on the project): name it by hand")

# the project library: the forecast project's own files (the same code), and loadtest.py
library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE), ("models.py", MODELS_CODE),
                        ("forecast.py", FORECAST_CODE), ("loadtest.py", LOADTEST_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)
print("Library: cashflow.py, warehouse.py, models.py, forecast.py (the forecast project's code) and loadtest.py")

# the flow, in 2 zones: Data prep (the fake data) and Load test (the test, its log and the report).
# Python recipes: (inputs, outputs, zone); load_test_log is a CSV; load_test_final, the dataset linked to the table of
# FINAL_LOAD = "sql" when it is not on Oracle (Oracle: no dataset; "csv": a file in the folder load_test_forecast)
final_dataset = FINAL_LOAD == "sql" and not final_oracle
FLOW = {"make_fake_data": ([], ["load_test_flows"], "Data prep"),
        "run_load_test": (["load_test_flows"], ["load_test_forecast", "load_test_log"]
                          + (["load_test_final"] if final_dataset else []), "Load test"),
        "load_test_report": (["load_test_log"], ["load_test_report"], "Load test")}
FOLDERS = ["load_test_flows", "load_test_forecast", "load_test_report"]
CSV = {"style": "excel", "charset": "utf8", "separator": ",", "quoteChar": '"', "escapeChar": "\\",
       "dateSerializationFormat": "ISO", "arrayMapFormat": "json", "parseHeaderRow": True, "compress": ""}
datasets = {d["name"] for d in project.list_datasets()}
recipes = {r["name"] for r in project.list_recipes()}
for old in ("run_test_plan",):                       # the earlier recipe, replaced by run_load_test
    if old in recipes:
        project.get_recipe(old).delete()
        recipes.discard(old)
existing_folders = {f["name"]: f["id"] for f in project.list_managed_folders()}
folders = {name: (project.get_managed_folder(existing_folders[name]) if name in existing_folders
                  else project.create_managed_folder(name, connection_name=OUTPUT_FOLDER_CONNECTION))
           for name in FOLDERS}
# the fake data is only made when asked by name (step 2 of the scenario), never by building what follows it
folder_settings = folders["load_test_flows"].get_settings()
folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
folder_settings.save()
variables = project.get_variables()                  # what run_load_test reads: the final load, csv or sql
variables["standard"].update(final_load=FINAL_LOAD, final_connection=FINAL_CONNECTION.strip(),
                             final_schema=FINAL_SCHEMA.strip(), final_table=FINAL_TABLE.strip())  # loadtest.final_table
variables["standard"].update(log_sql_connection=LOG_SQL_CONNECTION.strip(), log_sql_schema=LOG_SQL_SCHEMA.strip(),
                             log_sql_table=LOG_SQL_TABLE.strip())    # the SQL copy of load_test_log (log_to_sql)
project.set_variables(variables)
print(f"Final load of each test: {FINAL_LOAD}"
      + (f", the table {final_sql} on {FINAL_CONNECTION}" if final_sql else ""))
if LOG_SQL_CONNECTION.strip() and LOG_SQL_TABLE.strip():
    loadtest = sys.modules["loadtest"]
    log_sql = forecast.table_name(LOG_SQL_SCHEMA, LOG_SQL_TABLE)
    try:                                             # made now, so a missing right shows before the tests run
        print(f"SQL copy of the log: {loadtest.ensure_log_table(LOG_SQL_CONNECTION.strip(), log_sql)}")
    except Exception as e:
        print(f"WARNING: the SQL copy of the log, {log_sql} on {LOG_SQL_CONNECTION}, could not be made or read: {e}\n"
              "The tests still run (their rows go to the dataset load_test_log); the job run_load_test will fail at "
              "its end until the table can be written. To make it by hand (Oracle):")
        print(forecast.create_table_sql(log_sql, loadtest.LOG_COLUMNS))
else:
    print("SQL copy of the log: none (LOG_SQL_CONNECTION is empty)")
if final_dataset:   # load_test_final: a dataset linked to the table made above (not a managed one: no new table)
    schema, table = FINAL_SCHEMA.strip().lower(), FINAL_TABLE.strip().lower()  # as PostgreSQL stores unquoted names
    try:                                                                        # (Dataiku quotes them)
        kind = client.get_connection(FINAL_CONNECTION.strip()).get_info().get_type()
    except Exception:
        kind = "PostgreSQL"
    if "load_test_final" in datasets:
        raw = project.get_dataset("load_test_final").get_settings().get_raw()
        params = raw.get("params", {})
        linked = (params.get("connection"), params.get("table"), params.get("schema") or "")
        if raw.get("managed") or linked != (FINAL_CONNECTION.strip(), table, schema):
            raise RuntimeError(f"The dataset load_test_final is not linked to the table {final_sql} on "
                               f"{FINAL_CONNECTION} (for example the managed dataset of an older version, whose "
                               "table Dataiku named): delete it (Flow > load_test_final > Delete, without its data), "
                               "then run again.")
    if "load_test_final" not in datasets:
        project.create_sql_table_dataset("load_test_final", kind, FINAL_CONNECTION.strip(), table, schema)
    project.get_dataset("load_test_final").autodetect_settings().save()        # its columns, read from the table
    print(f"Dataset load_test_final: linked to the table {final_sql} on {FINAL_CONNECTION} (Dataiku's writer)")
if "load_test_log" not in datasets:
    settings = project.new_managed_dataset("load_test_log").with_store_into(OUTPUT_CONNECTION).create().get_settings()
    settings.get_raw()["formatType"] = "csv"
    settings.get_raw()["formatParams"].update(CSV)
    settings.save()
ref = lambda name: folders[name].id if name in folders else name
# first, the recipes whose inputs or outputs changed are removed (made again below), so no output is held by an old one;
# an output no longer wanted (load_test_final, for FINAL_LOAD = "csv") is only taken off the recipe (nothing to create)
for name, (inputs, outputs, _) in FLOW.items():
    if name in recipes:
        settings = project.get_recipe(name).get_settings()
        extra = set(settings.get_flat_output_refs()) - set(map(ref, outputs))
        if extra and set(settings.get_flat_output_refs()) - extra == set(map(ref, outputs)):
            for role in settings.get_recipe_outputs().values():
                role["items"] = [item for item in role.get("items", []) if item.get("ref") not in extra]
            settings.save()
            print("recipe", name, "no longer writes", ", ".join(sorted(extra)))
        if (set(settings.get_flat_input_refs()) != set(map(ref, inputs))
                or set(settings.get_flat_output_refs()) != set(map(ref, outputs))):
            project.get_recipe(name).delete()
            recipes.discard(name)
            print("recipe", name, "has new inputs or outputs: made again")
for name, (inputs, outputs, _) in FLOW.items():
    if name not in recipes:
        builder = project.new_recipe("python", name)
        for i in inputs:
            builder.with_input(ref(i))
        for o in outputs:
            builder.with_output(ref(o), append=o == "load_test_log")
        builder.create()
    settings = project.get_recipe(name).get_settings()
    settings.set_payload(RECIPES[name])
    settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    print("python recipe", name, "->", ", ".join(outputs))

# the zones: Dataiku's Default zone becomes Data prep (so there is no empty zone). It exists once another zone does
flow = project.get_flow()
existing_zones = {z.name: z for z in flow.list_zones()}
zones = {"Load test": existing_zones.get("Load test") or flow.create_zone("Load test")}
zones["Data prep"] = existing_zones.get("Data prep") or flow.get_default_zone()
if zones["Data prep"].name != "Data prep":
    zone_settings = zones["Data prep"].get_settings()
    zone_settings.name = "Data prep"
    zone_settings.save()
for _, outputs, zone_name in FLOW.values():
    for item in outputs:
        try:
            zones[zone_name].add_item(folders[item] if item in folders else project.get_dataset(item))
        except Exception as e:
            if "already" not in str(e).lower():
                raise
print("Flow ready: zones Data prep (the fake data) and Load test (the test, its log, the report)")

# the scenario, run by hand (no trigger). Its step 1 sets the tests (Scenarios > Run the load test > Steps > 1): the
# settings above when the scenario is made; running this notebook again keeps the tests already set there.
def numbers_step(label, variables):
    return {"id": "step_0", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
            "runConditionStatuses": ["SUCCESS", "WARNING"], "type": "define_vars",
            "params": {"variables": variables, "definitions": []}}


def build_step(n, label, items, even_if_failed=False):
    return {"id": f"step_{n}", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
            "runConditionStatuses": ["SUCCESS", "WARNING"] + (["FAILED"] if even_if_failed else []),
            "type": "build_flowitem", "params": {
                "builds": [{"type": "MANAGED_FOLDER", "itemId": folders[i].id, "partitionsSpec": ""} if i in folders
                           else {"type": "DATASET", "itemId": i, "partitionsSpec": ""} for i in items],
                "jobType": "NON_RECURSIVE_FORCED_BUILD", "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


SCENARIO = "Run the load test"
DESCRIPTION = (
    "Run by hand. Step 1 (Steps > 1. Set the tests): tests, run one after the other, a grid "
    '{"clients": [...], "months": ... (at least 16), "postings_per_month": ..., "clients_per_group": [...], '
    '"workers": [...]} (each a number or a list: every combination is a test) or a list of tests. '
    "It makes the fake data of each size if it is not there yet (zone Data prep, not timed, kept for the next runs), "
    "runs each test (the forecast, then its final load: a CSV file or the SQL table), then the report: each client count on its own, the settings "
    "compared at the same count (dashboard Load test, folder load_test_report), even when a test failed")
STEPS = [numbers_step("Set the tests", {"tests": TESTS}),
         build_step(1, "The fake data of each size (not timed)", ["load_test_flows"]),
         build_step(2, "The tests", ["load_test_log"]),
         build_step(3, "The report", ["load_test_report"], even_if_failed=True)]
existing_scenarios = {s["name"]: s["id"] for s in project.list_scenarios()}


def first_numbers(name):
    """The variables in step 1 of an existing scenario."""
    first = (project.get_scenario(existing_scenarios[name]).get_settings().get_raw()["params"].get("steps") or [{}])[0]
    return first.get("params", {}).get("variables", {}) if first.get("type") == "define_vars" else {}


# tests already set: kept (and the earlier version's single test, its numbers directly in step 1, as the only test)
before = {}
for name in ("Create the fake data", SCENARIO):
    if name in existing_scenarios:
        before.update({k.replace("fake_", ""): v for k, v in first_numbers(name).items()})
variables = STEPS[0]["params"]["variables"]
if isinstance(before.get("tests"), (list, dict)) and before["tests"]:
    variables["tests"] = before["tests"]
elif any(k in before for k in CHECKED[0]):
    variables["tests"] = [{k: before.get(k, v) for k, v in CHECKED[0].items()}]
for old in ("Create the fake data", "Run the test plan"):           # the earlier scenarios, now this one
    if old in existing_scenarios:
        project.get_scenario(existing_scenarios.pop(old)).delete()
        print(f"Scenario {old}: removed (its numbers are now in step 1 of {SCENARIO})")
if SCENARIO in existing_scenarios:
    settings = project.get_scenario(existing_scenarios[SCENARIO]).get_settings()
    settings.get_raw()["description"] = DESCRIPTION
    settings.get_raw()["params"]["steps"] = STEPS
    settings.save()
else:
    project.create_scenario(SCENARIO, "step_based", {"active": False, "description": DESCRIPTION,
                                                     "params": {"steps": STEPS}})
print(f"Scenario ready: {SCENARIO} (run it by hand). Step 1, Set the tests: {json.dumps(variables)}")
for n, test in enumerate(sys.modules["loadtest"].check_tests(variables["tests"]), 1):
    print(f"    test {n}: {json.dumps(test)}")

## 2. Run it

What the scenario **Run the load test** runs, with the tests of its step 1 (with many clients it takes long: the progress shows here).

In [ ]:
import time


def build(step, items, mode):
    """Runs a Dataiku job and waits for it, showing its PROGRESS lines here (also in the job's log) every 30 s."""
    print(f"\n{step} ...", flush=True)
    builder = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item in items:
        if item in folders:
            builder.with_output(folders[item].id, object_type="MANAGED_FOLDER")
        else:
            builder.with_output(item)
    job, shown, began = builder.start(), 0, time.time()
    while True:
        state = job.get_status().get("baseStatus", {}).get("state", "")
        try:
            lines = [l[l.index("PROGRESS ") + 9:] for l in job.get_log().splitlines() if "PROGRESS " in l]
        except Exception:
            lines = []
        for line in lines[shown:]:
            print("   ", line, flush=True)
        shown = max(shown, len(lines))
        if state in ("DONE", "FAILED", "ABORTED"):
            break
        time.sleep(30 if time.time() - began > 60 else 5)
    if state != "DONE":
        raise RuntimeError(f"{step}: the job {state}. Open it in Jobs (id {job.id}) to read its log.")
    print(f"{step}: done in {(time.time() - began) / 60:.1f} min", flush=True)



if RUN_NOW:      # what the scenario Run the load test runs, as jobs (the recipes read its step 1)
    build("1/3 The fake data of each size (not timed; kept if already there)", ["load_test_flows"],
          "NON_RECURSIVE_FORCED_BUILD")
    try:
        build("2/3 The tests", ["load_test_log"], "NON_RECURSIVE_FORCED_BUILD")
    finally:                                         # the report even when a test failed
        build("3/3 The report", ["load_test_report"], "NON_RECURSIVE_FORCED_BUILD")
        log = dataiku.Dataset("load_test_log").get_dataframe().sort_values("run_at")
        print("The last tests (the newest last):")
        display(log[["batch", "test", "status", "workers", "clients", "clients_per_group", "seconds_forecast",
                     "seconds_sql_load", "clients_per_second", "peak_memory_mb", "error"]].tail(10).set_index("batch"))
        print("The full report: Dashboards > Load test, or the folder load_test_report > latest_report.html")
else:
    print("Set up. Then: Scenarios > Run the load test (its tests: Steps > 1), Run")